# dialog

> dialog provides message and dialog introspection, and import/export utilities for solveit dialogs.

In [ ]:
#| default_exp dialog

In [ ]:
#| export
#| hide
import re, json, ast
from html import escape
from pathlib import Path
from typing import TypeAlias
from IPython import get_ipython
from mistletoe import Document
from mistletoe.block_token import CodeFence
from mistletoe.markdown_renderer import MarkdownRenderer
from nbdev.process import extract_directives
from fastcore.foundation import patch, adict, AttrDict as AD
from fastcore.meta import delegates
from fastcore.py2pyi import _body_ellip, functypes
from dialoghelper.core import add_msg, read_msg, find_msgs, msg_idx, run_msg, toggle_header, ast_py, find_dname
from pote.dutil import in_dialog

In [ ]:
#| hide
from fastcore.test import *
from fastcore.py2pyi import _proc_patches, has_deco
from nbdev.config import get_config, is_nbdev
from pote.dutil import link_msg

## message utils
Core primitives for working with solveit messages: a `Msg` type alias, utilities for reading nbdev directives and message metadata, progressive disclosure of note content, and an extended `MsgX` subclass with convenience properties.

### message tag

Defines `Msg` as a type alias for `adict`, providing a concise, readable type annotation for solveit message objects throughout the dialog.

In [ ]:
#| export
Msg: TypeAlias = AD

### message metadata & directives

Utilities for inspecting solveit message metadata. `is_exported` checks if a message is marked for export (via solveit metadata or nbdev `#|export`/`#|exports`/`#|exporti` directives). `is_hidden` checks if a message is hidden from doc generation via `#|hide`. `is_echo` checks if a code message is marked to show source via `#|echo`. `is_header` checks if the note message start with a markdown heading.

In [ ]:
cell = AD(source='#|exporti\nprint(13)')
dirs = extract_directives(cell, remove=False)
bool(dirs.keys() & {'exporti', 'exports', 'export'})

True

In [ ]:
print(extract_directives(AD(source='#| echo\nprint(13)'), remove=False))
print(extract_directives(AD(source='#| echo: true\nprint(13)'), remove=False))

{'echo': []}
{'echo:': ['true']}


In [ ]:
#| exporti
_exports = {'exporti', 'exports', 'export'}
_hidden = {'hide'}
_echo = {'echo', 'echo:'}

In [ ]:
#| export
def is_exported(m:Msg) -> bool:
    "Is message exported with solveit metadata or nbdev directive?"
    return m.is_exported or (m.msg_type == 'code' and bool(
        extract_directives(AD(source=m.content), remove=False).keys() & _exports))

def is_hidden(m:Msg) -> bool:
    "Is message hidden by a nbdev directive (only for doc generation, not AI visibility)?"
    return m.msg_type == 'code' and bool(
        extract_directives(AD(source=m.content), remove=False).keys() & _hidden)

def is_echo(m:Msg) -> bool:
    "Does message have an echo directive that isn't false?"
    if m.msg_type != 'code': return False
    d = {k:v for k,v in extract_directives(AD(source=m.content), remove=False).items() if k in _echo}
    return bool(d) and all('false' not in v for v in d.values())

def is_header(m:Msg) -> bool:
    "Does note message content start by a markdown header?"
    return m.msg_type == 'note' and m.content.startswith('#')

In [ ]:
test_is(is_exported(await read_msg()) and is_exported(await read_msg(-2)), True)
test_is(is_exported(await read_msg(-4)), False)

In [ ]:
#| echo
test_is(is_hidden(Msg(msg_type='code', content='#|hide\n\nsome content')), True)

test_is(is_header(Msg(msg_type='note', content='# a')), True)
test_is(is_header(Msg(msg_type='note', content=' ### a')), False)

test_is(is_echo(Msg(msg_type='code', content='#| echo\nprint(1)')), True)
test_is(is_echo(Msg(msg_type='code', content='#|hide\n#| echo: true\nprint(1)')), True)
test_is(is_echo(Msg(msg_type='code', content='#| echo: false\nprint(1)')), False)
test_is(is_echo(Msg(msg_type='note', content='#| echo\nsome text')), False)
test_is(is_echo(Msg(msg_type='code', content='print(1)')), False)

### notes disclosure

`clamp_note` supports four levels of **progressive disclosure** for markdown notes.
| Level | Returns |
|---|---|
| `'header'` | The first line, if it is a markdown header; otherwise `''` |
| `'header+'` | Header + first non-blank line after it |
| `'header++'` | Header + first paragraph, only if it immediately follows the header |
| `int` (e.g. `30`) | First *n* lines, regardless of structure |
All levels return `''` if the content is empty or (for header levels) has no leading header.

In [ ]:
_section = (await read_msg()).content

In [ ]:
#| export
def clamp_note(
    c:str, # Markdown note content
    level:str|int='header' # Disclosure level: 'header', 'header+', 'header++', or int for max lines
) -> str:
    "Return a progressively disclosed excerpt of a markdown note."
    c = c.strip().splitlines()
    if c:
        if isinstance(level, int): return '\n'.join(c[:level])
        if level == 'header': return c[0] if c[0][0] == '#' else ''
        if level == 'header+':
            i, res = 0, c[0]
            if res[0] != '#': return ''
            while (i:=i+1) < len(c):
                if c[i]: return f"{res}\n{c[i]}"
            return res
        if level.startswith('header++'):
            header = c[0] if c[0][0] == '#' else ''
            if not header: return ''
            blocks = [b.strip() for b in '\n'.join(c[1:]).split('\n\n') if b.strip()]
            if blocks:
                first = blocks[0].splitlines()[0]
                if not first.startswith('#') and not first.startswith(('```', '~~~')):
                    return f"{header}\n{blocks[0]}"
            return header
    return ''

In [ ]:
# Paragraph immediately follows header
test_eq(clamp_note('# hdr\n\nParagraph text.', 'header++'), '# hdr\nParagraph text.')

# Multi-line paragraph immediately follows header
test_eq(clamp_note('# hdr\n\nLine one.\nLine two.', 'header++'), '# hdr\nLine one.\nLine two.')

# First block is a subheader — return header only
test_eq(clamp_note('# hdr\n\n## subheader\n\nActual paragraph.', 'header++'), '# hdr')

# First block is a code fence — return header only
test_eq(clamp_note('# hdr\n\n```python\ncode\n```\n\nReal paragraph.', 'header++'), '# hdr')

# No content after header
test_eq(clamp_note('# hdr', 'header++'), '# hdr')

# No header
test_eq(clamp_note('just text\n\nParagraph.', 'header++'), '')

# Lines
test_eq(clamp_note('just text\n\nParagraph.', 30), 'just text\n\nParagraph.')
test_eq(clamp_note('a\n'*50, 30), ('a\n'*30)[:-1])

# Empty / whitespace
test_eq(clamp_note('', 'header++'), '')
test_eq(clamp_note('   ', 'header++'), '')

In [ ]:
c = _section
test_eq(clamp_note(c), '### notes disclosure')
test_eq(clamp_note(c, 'header+').splitlines()[1][:21], '`clamp_note` supports')
test_eq(clamp_note(c, 5).splitlines()[:5], c.splitlines()[:5])
print(clamp_note(c, 'header++'))

### notes disclosure
`clamp_note` supports four levels of **progressive disclosure** for markdown notes.
| Level | Returns |
|---|---|
| `'header'` | The first line, if it is a markdown header; otherwise `''` |
| `'header+'` | Header + first non-blank line after it |
| `'header++'` | Header + first paragraph, only if it immediately follows the header |
| `int` (e.g. `30`) | First *n* lines, regardless of structure |
All levels return `''` if the content is empty or (for header levels) has no leading header.


In [ ]:
#| echo
cnt = '#### test\n> Paragraph 1.\nLine.\n\nParagraph 2\n```python\na=13\n```'
test_eq(clamp_note(cnt, 'header'), '#### test')

test_eq(clamp_note(cnt, 'header+'), '#### test\n> Paragraph 1.')
test_eq(clamp_note(cnt, 'header++'), '#### test\n> Paragraph 1.\nLine.')

### message extended

Extends the base `Msg` (`adict`) type with `MsgX`, a subclass that adds convenience properties for inspecting message metadata.
`msgx()` casts any `Msg` in-place to `MsgX` (zero-copy, via `__class__` reassignment). `MsgX` then exposes `.exported_x` and `.hidden_x` as properties wrapping `is_exported`/`is_hidden`.


In [ ]:
import fastcore.basics
from fastcore.basics import adict
adict??
AD??


```python
class adict(dict):
    "`dict` subclass that also provides access to keys as attrs"
    def __getattr__(self,k): return self[k] if k in self else stop(AttributeError(k))
    def __setattr__(self, k, v): (self.__setitem__,super().__setattr__)[k[0]=='_'](k,v)
    def __dir__(self): return super().__dir__() + list(self.keys())
```

**File:** `/usr/local/lib/python3.12/site-packages/fastcore/basics.py`


```python
class AttrDict(adict):
    "`dict` subclass that also provides access to keys as attrs, and has a pretty markdown repr"
    def _repr_markdown_(self): return f'```python\n{pprint.pformat(self, indent=2)}\n```'
    def copy(self): return AttrDict(**self)
```

**File:** `/usr/local/lib/python3.12/site-packages/fastcore/basics.py`

In [ ]:
msg = await read_msg()
test_is(isinstance(msg, adict), True)
test_is(isinstance(msg, Msg), False)
msg

{'idle_evt': '<asyncio.locks.Event object at 0x7a1f6bc706e0 [set]>',
 'id': '_eb1a767c',
 'time_run': '2026-05-09T13:32:21.104362+00:00',
 'is_exported': 0,
 'skipped': 1,
 'bookmark': None,
 'i_collapsed': 0,
 'o_collapsed': 0,
 'heading_collapsed': False,
 'i_clamp': False,
 'o_clamp': False,
 'pinned': 0,
 'run': None,
 'oob': None,
 'order': 49,
 'content': 'import fastcore.basics\nfrom fastcore.basics import adict\nadict??\nAD??',
 'output': '\n```python\nclass adict(dict):\n    "`dict` subclass that also provides access to keys as attrs"\n    def __getattr__(self,k): return self[k] if k in self else stop(AttributeError(k))\n    def __setattr__(self, k, v): (self.__setitem__,super().__setattr__)[k[0]==\'_\'](k,v)\n    def __dir__(self): return super().__dir__() + list(self.keys())\n```\n\n**File:** `/usr/local/lib/python3.12/site-packages/fastcore/basics.py`\n\n```python\nclass AttrDict(adict):\n    "`dict` subclass that also provides access to keys as attrs, and has a pretty mark

In [ ]:
#| export
class MsgX(Msg): ...

In [ ]:
msg.__class__ = MsgX
test_is(isinstance(msg, MsgX), True)

In [ ]:
#| export
def msgx(msg:adict) -> None:
    "Return `msg` cast to MsgX."
    msg.__class__ = MsgX
    return msg

In [ ]:
#| export
@patch(as_prop=True)
def exported_x(self:MsgX) -> bool: return is_exported(self)
@patch(as_prop=True)
def hidden_x(self:MsgX) -> bool: return is_hidden(self)
@patch(as_prop=True)
def echo_x(self:MsgX) -> bool: return is_echo(self)
@patch(as_prop=True)
def header_x(self:MsgX) -> bool: return is_header(self)

What does `msgx` do? Critique.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`msgx` casts an `adict` message to `MsgX` in-place by reassigning `__class__`, then returns it. This is a fast, zero-copy type "upgrade" that lets you use `MsgX` properties/methods on an existing dict without creating a new object.

**Critique:**

- The in-place mutation is a side effect that the name and docstring don't make obvious — "cast in-place" hints at it, but callers might be surprised their original `msg` is also now a `MsgX`
- The type annotation `msg:adict` is slightly misleading since it'll work on any dict subclass (e.g. plain `adict`, `AD`, etc.)
- No guard against passing a `msg` that's already a `MsgX` — harmless, but worth knowing
- It's essentially a one-liner that could be inlined wherever needed; whether it deserves its own function depends on how often it's called

<details class='token-usage-details'><summary>$0.0215</summary>

`total=13,588 | in=13,365 | out=223 | cached=80.7% | $0.0215`

</details>


### extended `find_msgs`
`find_msgs` returning MsgX.

In [ ]:
#| export
@delegates(find_msgs)
async def find_msgs_x(*args, **kwargs) -> list[MsgX]:
    "find_msgs returning MsgX instances."
    ms = await find_msgs(*args, **kwargs)
    for m in ms: m.__class__ = MsgX
    return ms

## dialog utils
Utilities for treating dialogs as first-class objects: resolving their backing files and Python modules, and introspecting runtime state such as cell outputs.

### dialog path
Resolves the filesystem path of a solveit dialog's backing `.ipynb` notebook file.

In [ ]:
#| export
def nb_path(dname:str=''): return (Path.home()/(dname or find_dname()).removeprefix('/')).with_suffix('.ipynb')

In [ ]:
_ = nb_path()
test_is(_.exists(), True)
_

Path('/app/data/prj/pote/nbs/01_dialog.ipynb')

In [ ]:
test_eq(nb_path('/a/b/c'), Path.home()/'a/b/c.ipynb')

### dialog module
Resolves the nbdev module name declared in a dialog or solo module if not found, enabling other tools to know which Python module a dialog exports to (if using `dlg_export`).

In [ ]:
#| export
_exp_pat =r'^\s*#\s*\|\s*default_exp\s+(\w+)'
async def dlg_module(dname:str='', solo_nb:bool=False):
    "Finds the nbdev default_exp in a dialog, or the solo module name if not nbdev"
    from nbdev.config import is_nbdev
    if not is_nbdev(p := nb_path(dname)) or solo_nb: return p.stem
    msgs = await find_msgs(_exp_pat, dname=(dname or find_dname()), limit=1, include_output=False, include_meta=False)
    if msgs: return msgs[0].content.strip().split()[-1]

In [ ]:
test_eq(await dlg_module(), 'dialog')
test_eq(await dlg_module('/a/b/c'), 'c')

### dialog export
Triggers nbdev export for a dialog, compiling its exported cells into the corresponding Python module on disk.

In [ ]:
#| export
async def dlg_export(dname:str='', solo_nb:bool=False):
    "Export dialog `dname` to Python module"
    from nbdev.export import nb_export
    from nbdev.config import get_config, is_nbdev
    p = nb_path(dname)
    if not (_is := is_nbdev(p)): solo_nb = True
    mod = await dlg_module(dname, solo_nb=solo_nb)
    lib_path = p.parent if solo_nb else get_config(p).lib_path if _is else Path('.')
    if solo_nb: nb_export(p, lib_path=lib_path, name=mod, solo_nb=solo_nb)
    else: nb_export(p)
    return (lib_path/f"{mod}.py").exists()

In [ ]:
mod_p = (get_config().lib_path/(await dlg_module())).with_suffix('.py')
if mod_p.exists(): mod_p.unlink()

test_is(await dlg_export(), True)
test_is(mod_p.exists(), True)

### dialog link
Generates a clickable HTML hyperlink to any solveit dialog by path, for use in notes and outputs.

In [ ]:
#| export
def get_dialog_link(
    path:str # Path to dialog (e.g. '/aai-ws/dialoghelper/nbs/00_core')
) -> str:
    "Return an HTML link to open a dialog in Solveit."
    from dialoghelper.core import dialog_link
    return dialog_link(path).data

In [ ]:
get_dialog_link('/prj/dutil/nbs/00_dutil')

'<a href="/dialog_?name=prj%2Fdutil%2Fnbs%2F00_dutil" target="_blank">prj/dutil/nbs/00_dutil</a>'

### empty .ipynb
Empty .ipynb file.

In [ ]:
#| export
def empty_dialog_nb() -> str:
    "Return a minimal dialog notebook json"
    return '''
{
"cells": [],
"metadata": {
"solveit_dialog_mode": "learning",
"solveit_ver": 2
},
"nbformat": 4,
"nbformat_minor": 5
}
'''

### message id
Exposes the id of the currently executing message, enabling code to self-reference its own position in the dialog.

In [ ]:
#| export
async def get_msg_id():
    "Get current message id."
    return (await read_msg(0)).id

In [ ]:
test_eq(await get_msg_id(), '_4de8f279')

### get output
Retrieves the rendered output string of any message by id, enabling programmatic inspection of what a cell produced.

In [ ]:
msgid = (msg := await read_msg()).id
test_eq(msg.output, '')

In [ ]:
2;

In [ ]:
msgid = (msg := await read_msg()).id
test_eq(msg.output, '')

In [ ]:
2

2

In [ ]:
msgid = (msg := await read_msg()).id
test_eq(msg.output, '2')

In [ ]:
display(HTML('asdf'))

HTML(asdf)

In [ ]:
msgid = (msg := await read_msg()).id
test_eq(msg.output, 'asdf')

In [ ]:
display(HTML('<ul><li>one</li></ul>'))

HTML(<ul><li>one</li></ul>)

In [ ]:
msgid = (msg := await read_msg()).id
test_eq(msg.output, '<ul><li>one</li></ul>')

In [ ]:
display(Markdown('### hdr3\n\n**bold**'))

<div class="prose">

### hdr3

**bold**

</div>

In [ ]:
msgid = (msg := await read_msg()).id
test_eq(msg.output, '<div class="prose">\n\n### hdr3\n\n**bold**\n\n</div>')

a note

In [ ]:
msgid = (msg := await read_msg()).id
test_eq(msg.output, '')

a prompt

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

a response

In [ ]:
promptid = (msg := await read_msg()).id
test_eq(msg.output, 'a response')

In [ ]:
msgid = (msg := await read_msg()).id
test_eq(msg.output, '')

In [ ]:
#| export
@delegates(read_msg)
async def get_output(id:str=None, **kwargs) -> list[str]:
    msg = await read_msg(0, id=id, **kwargs)
    return msg.output

In [ ]:
test_eq(await get_output(promptid), 'a response')

In [ ]:
FC.noop?

```python
def noop(
    x:NoneType=None, args:VAR_POSITIONAL, kwargs:VAR_KEYWORD
):

```



```
Do nothing
```



**File:** `/usr/local/lib/python3.12/site-packages/fastcore/imports.py`

**Type:** function

In [ ]:
(await read_msg()).output

'```python\ndef noop(\n    x:NoneType=None, args:VAR_POSITIONAL, kwargs:VAR_KEYWORD\n):\n\n```\n\n\n\n```\nDo nothing\n```\n\n\n\n**File:** `/usr/local/lib/python3.12/site-packages/fastcore/imports.py`\n\n**Type:** function'

### output formatting
Converts Python objects into a JSON-encoded display payload — capturing all available MIME representations (plain text, HTML, markdown) — for programmatic use outside the normal cell output pipeline.

In [ ]:
#| exporti
_format = get_ipython().display_formatter.format if in_dialog() else None

In [ ]:
_format(...)

({'text/plain': 'Ellipsis'}, {})

In [ ]:
_format(HTML('a'))

({'text/plain': 'HTML(a)', 'text/html': 'a'}, {})

In [ ]:
_format(Div('a'))

({'text/plain': '<div>a</div>',
  'text/html': '<div>a</div>',
  'text/markdown': '```html\n<div>a</div>\n\n```'},
 {})

In [ ]:
#| export
delegates(_format)
def format_output(o, **kwargs):
  d, md = _format(o, **kwargs) if _format else ('', {})
  return json.dumps([{"data": d, "metadata": md, "output_type": "display_data"}])

In [ ]:
format_output('')

'[{"data": {"text/plain": "\'\'"}, "metadata": {}, "output_type": "display_data"}]'

In [ ]:
format_output('a')

'[{"data": {"text/plain": "\'a\'"}, "metadata": {}, "output_type": "display_data"}]'

In [ ]:
format_output(Pre('a\nb\nc'))

'[{"data": {"text/plain": "<pre>a\\nb\\nc</pre>", "text/html": "<pre>a\\nb\\nc</pre>", "text/markdown": "```html\\n<pre>a\\nb\\nc</pre>\\n```"}, "metadata": {}, "output_type": "display_data"}]'

In [ ]:
format_output(Div('a'), exclude=['text/html'])

'[{"data": {"text/plain": "<div>a</div>", "text/markdown": "```html\\n<div>a</div>\\n\\n```"}, "metadata": {}, "output_type": "display_data"}]'

### markdown splitter
Splits a markdown string into its constituent blocks, separating fenced code blocks from surrounding prose, so callers can process or transform each block type independently.

In [ ]:
def split_md_blocks(md):
    "Split markdown string into list of fenced and non-fenced block strings"
    return [b for b in re.split(r'(^```\w*\n.*?^```)', md, flags=re.DOTALL|re.MULTILINE) if b.strip()]

In [ ]:
for _ in split_md_blocks("# Hello\n\nSome text\n\n```python\nprint('hi')\n```\n\nMore text"): print(_, '\n---')

# Hello

Some text

 
---
```python
print('hi')
``` 
---


More text 
---


The idiomatic mistletoe way is to walk the parsed AST and use a renderer for non-code blocks:

In [ ]:
def split_md_blocks(md):
    "Split markdown into fenced/non-fenced blocks via mistletoe AST"
    with HtmlRenderer() as renderer:
        doc = Document(md)
        return [f'```{c.language}\n{c.children[0].content}```' if isinstance(c, CodeFence)
                else renderer.render(c) for c in doc.children]

In [ ]:
split_md_blocks("# Hello\n\nSome text\n\n```python\nprint('hi')\n```\n\nMore text")

['<h1>Hello</h1>',
 '<p>Some text</p>',
 "```python\nprint('hi')\n```",
 '<p>More text</p>']

The key ideas here are: `Document` parses into an AST whose `.children` are block-level tokens; `CodeFence` tokens carry `.language` and their code content nested as `children[0].content`; everything else gets rendered to HTML via the renderer context. This gives you a starting point to explore other token types like `Heading`, `Paragraph`, `List`, etc.

We can swap to `MarkdownRenderer` which round-trips tokens back to raw markdown.

In [ ]:
def split_md_blocks(md):
    "Split markdown into fenced/non-fenced blocks via mistletoe AST"
    with MarkdownRenderer() as renderer:
        doc = Document(md)
        return [renderer.render(c) for c in doc.children]

In [ ]:
split_md_blocks("# Hello\n\nSome text\n\n```python\nprint('hi')\n```\n\nMore text")

['# Hello\n',
 '\n',
 'Some text\n',
 '\n',
 "```python\nprint('hi')\n```\n",
 '\n',
 'More text\n']

Collapse consecutive non-fenced code blocks.

In [ ]:
#| export
def split_md_blocks(md):
    "Split markdown into fenced/non-fenced blocks via mistletoe AST"
    with MarkdownRenderer() as renderer:
        doc = Document(md)
        blocks,buf = [],[]
        for c in doc.children:
            if isinstance(c, CodeFence):
                if buf: blocks.append(''.join(buf)); buf = []
                blocks.append(renderer.render(c))
            else: buf.append(renderer.render(c))
        if buf: blocks.append(''.join(buf))
        return blocks

In [ ]:
test_eq(split_md_blocks("# Hello\n\nSome text\n\n```python\nprint('hi')\n```\n\nMore text"),
    ['# Hello\n\nSome text\n\n', "```python\nprint('hi')\n```\n", '\nMore text\n'])

test_eq(split_md_blocks("No code blocks at all."), 
        ['No code blocks at all.\n'])

test_eq(split_md_blocks("```python\nonly_code()\n```"), 
        ['```python\nonly_code()\n```\n'])

test_eq(split_md_blocks("Before\n```python\ncode1()\n```\n```bash\ncode2\n```\nAfter"), 
        ['Before\n', '```python\ncode1()\n```\n', '```bash\ncode2\n```\n', 'After\n'])

test_eq(split_md_blocks("Text\n```\nno language\n```\nMore"), 
        ['Text\n', '```\nno language\n```\n', 'More\n'])

test_eq(split_md_blocks("A\n```python\ncode\n```\nB\n```js\nmore\n```\nC"), 
        ['A\n', '```python\ncode\n```\n', 'B\n', '```js\nmore\n```\n', 'C\n'])

## distill
Utilities for reading and writing message content across sessions.

### filter messages v0
Filters a list of messages down to those relevant for export or documentation: exported (non-hidden) code cells, note headers, and cells with an echo directive.

In [ ]:
#| echo
async def filter_msgs_v0(
    msgs:list[Msg]=None, # List of messages to filter or None to use all messages.
    dname:str='', # Dialog to get messages for; defaults to current dialog.
) -> list[MsgX]:
    "Return notes, exported messages not hidden and code cells that have echo directive."
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    return [m for m in msgs if (is_exported(m) and not is_hidden(m)) or is_header(m) or is_echo(m)]

In [ ]:
test_export_dname = find_dname('data/test_export')
test_export_dname

'/prj/pote/nbs/data/test_export'

In [ ]:
test_msgs = await find_msgs_x(dname=test_export_dname, include_skipped=True)

In [ ]:
fed_test_msgs = await filter_msgs_v0(test_msgs)
test_eq(len(fed_test_msgs), 9)

In [ ]:
msgs = await find_msgs_x(include_skipped=True)
first_idx, last_idx = await msg_idx('_cf210ba7'), await msg_idx('_842dc9a8')

In [ ]:
dlog_msgs = msgs[first_idx:last_idx]  # messages in main section
len(dlog_msgs), len(await filter_msgs_v0(dlog_msgs))

(248, 76)

### digest dialog
Format a filtered digest of dialog messages for AI consumption. Supports `xml` (preferred), `md`, and `json` formats. Headers are progressively disclosed via `clamp_note` to keep digests compact. YAML frontmatter is prepended via `mk_frontmatter`.


In [ ]:
#| export
_pmark = '**🤖Reply🤖**<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->\n'

In [ ]:
#| export
def _mk_tag(m, mtyp, ids):
    # parts = ([f'#{m.id}'] if ids else []) + [mtyp] + (['echo'] if echo else []) + (['skipped'] if m.skipped else []) + (['pinned'] if m.pinned else [])
    parts = ([f'#{m.id}'] if ids else []) + [mtyp]
    return ' '.join(parts)

In [ ]:
#| export
def output2md(c, langid='output'):
    return f'''```{langid}\n{c}\n```''' if c else ''

In [ ]:
test_eq(_mk_tag(MsgX(id='a'), 'code', ['a']), '#a code')

In [ ]:
#| export
def strip_directives(c): return ('\n'.join(l for l in c.splitlines() if not l.startswith('#|'))).strip()

In [ ]:
#| export
_striptag_pats = (
    re.compile(r'^#\|\W*\w+: \w+\W*$\n?', re.MULTILINE),
    re.compile(r'^<!-- \w+: \w+ -->\s*\n?', re.MULTILINE))

def strip_tags(c):
    for pat in _striptag_pats: c = pat.sub('', c)
    return c.strip()

In [ ]:
#| export
def _xml_msg(m:MsgX, ids, output, hdr_clamp):
    mtyp = m.msg_type
    c, attrs = '', (f' id="{m.id}"' if ids else '') + f' type="{mtyp}"'
    match mtyp:
        case 'note':
            clamp = not m.exported_x and m.header_x
            c = escape(strip_tags(clamp_note(m.content, hdr_clamp) if clamp else m.content))
            if c: c = f'<content>{c}</content>'
        case 'code':
            src = escape(strip_directives(m.content))
            out = f'\n  <output>{escape(m.output)}</output>' if output and m.output else ''
            if src or out: c = f'\n  <source>{src}</source>{out}\n'
        case 'prompt':
            p = escape(strip_tags(strip_directives(m.content)))
            r = f'\n  <response>{escape(m.output)}</response>' if m.output else ''
            if p or r: c = f'<prompt>{p}</prompt>{r}'
        case 'raw':
            c = escape(strip_tags(strip_directives(m.content)))
            if c: c = f'<msg{attrs}><content>{c}</content></msg>'
    return f'<msg{attrs}>{c}</msg>' if c else ''

In [ ]:
#| export
def _md_msg(m, ids, output, hdr_clamp):
    mtyp, c = m.msg_type, None
    match mtyp:
        case 'note':
            clamp = not m.exported_x and m.header_x
            c = strip_tags(clamp_note(m.content, hdr_clamp) if clamp else m.content)
        case 'code':
            c = f'```python\n{strip_directives(m.content)}\n```'
            if output and m.output: c += f'\n{output2md(m.output)}'
        case 'prompt':
            c = strip_tags(strip_directives(m.content))
            if output and m.output: c += f'\n{output2md(m.output, 'response')}'
        case 'raw': c = strip_tags(strip_directives(m.content))
    return f'<!-- {_mk_tag(m, mtyp, ids)} -->\n{c}\n' if c else ''

In [ ]:
#| export
def _json_msg(m, expd, echo, ids, output, hdr_clamp):
    mtyp = m.msg_type
    d = dict(type=mtyp)
    if ids: d['id'] = m.id
    match mtyp:
        case 'note':
            clamp = not m.exported_x and m.header_x
            d['content'] = strip_tags(clamp_note(m.content, hdr_clamp) if clamp else m.content)
        case 'code':
            d['source'] = strip_directives(m.content)
            if output and m.output: d['output'] = m.output
        case 'prompt':
            d['prompt'] = strip_tags(strip_directives(m.content))
            d['response'] = m.output
        case 'raw': d['content'] = strip_tags(strip_directives(m.content))
    return d

In [ ]:
#| export
_convs = dict(
    xml="msg elements have id and type attributes; source/output are child elements of code msgs; prompt/response are child elements of prompt msgs",
    json="msgs is a list of objects with type and id fields; code objects have source and output fields; prompt objects have prompt and response fields; others have content",
    md="<!-- #<msg_id> <msg_type> --> precedes each block; ```output blocks contain cell output; ```response blocks contain model response")

In [ ]:
#| export
def mk_frontmatter(path='', description='', n_msgs=0, n_exported=0, fmt='xml'):
    from datetime import datetime, timezone
    return ("---\n"
"digest: solveit-dialog\n"
f"path: {path}\n"
f"exported_at: {datetime.now(timezone.utc).isoformat(timespec='seconds')}\n"
f"description: {description!r}\n"
f"stats: {{total: {n_msgs}, exported: {n_exported}}}\n"
f"conventions: '{_convs[fmt]}'\n"
"---\n"
)

In [ ]:
#| export
def exportmsgs(
    msgs:list[Msg],
    hdr_clamp='header+',  # header level
    output=True,  # include code messages output
    ids=True,  # include message ids in cell comments
    frontmatter:dict=None,  # if present, call `mk_frontmatter` with this dict as args
    fmt='xml',  # 'xml'|'md'|'json'
):
    "Export messages as a digest for AI consumption"
    fn = {'xml': _xml_msg, 'md': _md_msg, 'json': _json_msg}[fmt]
    parts = [p for m in msgs if (p := fn(m, ids, output, hdr_clamp))]
    if fmt == 'json': res = json.dumps({'msgs': parts}, indent=2)
    else:
        body = '\n'.join(parts)
        res = f'<digest>\n{body}\n</digest>' if fmt=='xml' else body
    if frontmatter:
        frontmatter.update(n_exported=len(parts))
        fm = mk_frontmatter(**frontmatter)
    return f"{fm}\n{res}" if frontmatter else res

In [ ]:
fmt = 'xml'
s = exportmsgs(test_msgs, hdr_clamp='header++', fmt=fmt, frontmatter=dict(
    path=test_export_dname, description='A digest for testing purposes', n_msgs=len(test_msgs), fmt=fmt))

print(len(s))
print(s)

1897
---
digest: solveit-dialog
path: /prj/pote/nbs/data/test_export
exported_at: 2026-04-24T08:54:12+00:00
description: 'A digest for testing purposes'
stats: {total: 15, exported: 15}
conventions: 'msg elements have id and type attributes; source/output are child elements of code msgs; prompt/response are child elements of prompt msgs'
---

<digest>
<msg id="_b6147aa6" type="note"><content>##### Header (exported)</content></msg>
<msg id="_fad144f0" type="note"><content>##### Header w/ paragraph (exported)
Paragraph.
Line 2.</content></msg>
<msg id="_2752a761" type="note"><content>##### Note with code fence (exported, only header)</content></msg>
<msg id="_cdfea4f6" type="note"><content>Note w/out header (not exported)</content></msg>
<msg id="_4e4b6f65" type="code">
  <source># exported code
a = 13; print(a+1)</source>
  <output>14</output>
</msg>
<msg id="_fde18916" type="code">
  <source># exported code, echo
test_eq(a, 13)</source>
</msg>
<msg id="_44d67d56" type="code">
  <source

In [ ]:
fmt = 'xml'
s = exportmsgs(fed_test_msgs, hdr_clamp='header++', fmt=fmt, frontmatter=dict(
    path=test_export_dname, description='A digest for testing purposes', n_msgs=len(test_msgs), fmt=fmt))

print(len(s))
print(s)

1292
---
digest: solveit-dialog
path: /prj/pote/nbs/data/test_export
exported_at: 2026-04-24T08:54:12+00:00
description: 'A digest for testing purposes'
stats: {total: 15, exported: 9}
conventions: 'msg elements have id and type attributes; source/output are child elements of code msgs; prompt/response are child elements of prompt msgs'
---

<digest>
<msg id="_b6147aa6" type="note"><content>##### Header (exported)</content></msg>
<msg id="_fad144f0" type="note"><content>##### Header w/ paragraph (exported)
Paragraph.
Line 2.</content></msg>
<msg id="_2752a761" type="note"><content>##### Note with code fence (exported, only header)</content></msg>
<msg id="_4e4b6f65" type="code">
  <source># exported code
a = 13; print(a+1)</source>
  <output>14</output>
</msg>
<msg id="_fde18916" type="code">
  <source># exported code, echo
test_eq(a, 13)</source>
</msg>
<msg id="_868ac4bd" type="code">
  <source># exported, directive
c = 99</source>
</msg>
<msg id="_a607681b" type="prompt"><prompt>a p

In [ ]:
fmt = 'md'
s = exportmsgs(fed_test_msgs, hdr_clamp='header++', fmt=fmt, frontmatter=dict(
    path=test_export_dname, description='A digest for testing purposes', n_msgs=len(test_msgs), fmt=fmt))

print(s)

---
digest: solveit-dialog
path: /prj/pote/nbs/data/test_export
exported_at: 2026-04-24T08:54:12+00:00
description: 'A digest for testing purposes'
stats: {total: 15, exported: 9}
conventions: '<!-- #<msg_id> <msg_type> --> precedes each block; ```output blocks contain cell output; ```response blocks contain model response'
---

<!-- #_b6147aa6 note -->
##### Header (exported)

<!-- #_fad144f0 note -->
##### Header w/ paragraph (exported)
Paragraph.
Line 2.

<!-- #_2752a761 note -->
##### Note with code fence (exported, only header)

<!-- #_4e4b6f65 code -->
```python
# exported code
a = 13; print(a+1)
```
```output
14
```

<!-- #_fde18916 code -->
```python
# exported code, echo
test_eq(a, 13)
```

<!-- #_868ac4bd code -->
```python
# exported, directive
c = 99
```

<!-- #_a607681b prompt -->
a prompt (exported)
```response
a response
```

<!-- #_e1f61bec raw -->
RAW (exported)

<!-- #_55d40474 raw -->
EXPORTED RAW CONTENT



In [ ]:
fmt = 'xml'
s = exportmsgs(await filter_msgs_v0(dlog_msgs), hdr_clamp='header++', fmt=fmt, frontmatter=dict(
    path=find_dname(), description='pote.dutil dialog', n_msgs=len(msgs), fmt=fmt))

print(len(s))
print(s[:2000])

33550
---
digest: solveit-dialog
path: /prj/pote/nbs/00_dialog
exported_at: 2026-04-24T08:54:13+00:00
description: 'pote.dutil dialog'
stats: {total: 252, exported: 76}
conventions: 'msg elements have id and type attributes; source/output are child elements of code msgs; prompt/response are child elements of prompt msgs'
---

<digest>
<msg id="_cf210ba7" type="note"><content># dialog
&gt; dialog provides dialog introspection, message linking, tool discovery, and import/export utilities for solveit dialogs.</content></msg>
<msg id="_279f4bb3" type="note"><content>## message utils
Core primitives for working with solveit messages: a `Msg` type alias, utilities for reading nbdev directives and message metadata, progressive disclosure of note content, and an extended `MsgX` subclass with convenience properties.</content></msg>
<msg id="_e229f022" type="note"><content>### message tag
Defines `Msg` as a type alias for `adict`, providing a concise, readable type annotation for solveit message

#| export
Format a filtered, clamped digest of dialog messages for AI consumption in other sessions. Three output formats are supported: `xml` (preferred — unambiguous structure), `md`, and `json`, each implemented by a parallel `_xml_msg`/`_md_msg`/`_json_msg` function dispatched by `exportmsgs`.

**Clamping:** note content is progressively disclosed via `clamp_note(level)` — from header-only to header+paragraph — unless the note is fully exported. This keeps digests compact while preserving section structure and intent.

**Frontmatter:** `mk_frontmatter` prepends YAML metadata (path, timestamp, stats, format conventions) so the consuming LLM knows what it's looking at.

**Next.**
Isolate sections and filtered list of messages. Two modes:
- gather list of messages and export as is
- gather list of messages and dependencies (real isolation) so if written in a new dialog will execute correctly. See isolation dialog exploration.

### filter messages v1

In [ ]:
(await read_msg(0, id='_8ca984e4')).content

"#| exporti\n_exports = {'exporti', 'exports', 'export'}\n_hidden = {'hide'}\n_echo = {'echo', 'echo:'}"

In [ ]:
msgs = await find_msgs(only_exp=True)
'_8ca984e4' in {_.id for _ in msgs}

False

#| export
`find_msgs` is a staple of SolveIt, a very powerful introspection tool. But it have some limitations (see above how it doesn't honor explicit `nbdev` export directives) and I need more search granularity (e.g., to improve `exportmsgs`).

We're going to implement `filter_msgs_v1`, a wrapper over `find_msgs_x` with progressively improved search semantics. With no arguments, `filter_msgs_v1` return an empty list.

```python
def filter_msgs_v1(*args, **kwargs) -> list[MsgX]: ...
```

Let's start with `nbdev` export directives: if given `exported=True`, `filter_msgs_v1` should return **all** exported messages, those marked with SolveIt UI toggle **and** the ones with `nbdev` exported directives.

In [ ]:
async def filter_msgs_v1(
    msgs:list[Msg]=None, # List of messages to filter or None to use all messages.
    exported:bool=False,
    dname:str='', # Dialog to get messages for; defaults to current dialog.
) -> list[MsgX]:
    "Filter messages with improved search semantics over `find_msgs`"
    if not exported: return []
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    if exported: msgs = [m for m in msgs if is_exported(m)]
    return msgs

In [ ]:
ms = await filter_msgs_v1(exported=True)
'_8ca984e4' in {m.id for m in ms}

True

As always, you jumped right into programming. I prefer to analyze the problem a bit before I start writing code.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You're right, sorry! Let's think through it.

`find_msgs` has two limitations we're addressing:

1. **Export detection** — `only_exp=True` only sees the solveit UI toggle, missing `#| export*` directives. We've handled this with `exported=True`.

2. **Search granularity** — the `**Next**` note hints at two upcoming features: filtering by *section* (a header + its children), and filtering by *dependencies* (messages a given cell needs to run correctly in isolation).

Before we add more to `filter_msgs_v1`, it's worth asking: what's the next concrete gap you're running into with `filter_msgs_v0`/`exportmsgs`? That'll tell us which search primitive to tackle next — section filtering, dependency tracking, or something else entirely.

That's getting ahead of ourselves. I have no idea what the final version of `filter_msgs_v1` will be like. We'll refine it little by little, adding one feature or addressing one need at a time.

In this stage we know four things: one, no argument, empty result; two, thus, unlike `find_msgs_x`, the given arguments will probably be cumulative, add to the result, instead of substract from it; three, the first filter to implement, `exported`; four, next filters? well, we have the example of `filter_msgs_v0`/`exportmsgs` which shows a few more: notes, echo cells, etc. so, we can give ourselves an intermediate goal: refactor the filter logic from `filter_msgs_v0` into `filter_msgs_v1`.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good framing. So the key design insight is **union semantics**: each flag adds a set of messages to the result, and the final result is their union (deduplicated, preserving order).

Looking at `filter_msgs_v1`, the inclusion predicate is:

```python
if (is_exported(m) and not is_hidden(m)) or is_header(m) or is_echo(m)
```

That maps cleanly to flags for `filter_msgs_v1`:

- `exported=True` and `hidden=False`
- `headers=True` — all header messages
- `echo=True` — code cells with `#| echo: true`

Yes, off the top of my head, one thing come to mind regarding those arguments: instead of listing all possible arguments, let's group them in logical categories: metadata=['exported', ...], msg_type=['code', 'note', ...]

So, `msg_type='note'` would include all notes; `msg_type=['note','code']` would include all note and code messages. `metadata='exported` would include all messages marked as exported; `metadata=['exported','echo']` would include all messages marked as exported or echoed. etc. For now, the semantics of the "'" is **and** ('note','code' really means 'note' and 'code') but perhaps will add additional semantics in the future.

So `msg_type` and `metadata` are the two category axes, each accepting a string or list, with union semantics across and within categories. Let's define the metadata predicate map and rewrite `filter_msgs_v1`:


In [ ]:
#| export
_meta_fns = dict(exported=is_exported, echo=is_echo, header=is_header, hidden=is_hidden)

In [ ]:
async def filter_msgs_v1(
    msgs:list[Msg]=None,
    msg_type:str|list=None,
    metadata:str|list=None,
    exclude_metadata:str|list=None,
    dname:str='',
) -> list[MsgX]:
    "Filter messages with union semantics across msg_type and metadata flags."
    if not msg_type and not metadata: return []
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    mt = ([msg_type] if isinstance(msg_type, str) else msg_type) or []
    md = ([metadata] if isinstance(metadata, str) else metadata) or []
    ex = ([exclude_metadata] if isinstance(exclude_metadata, str) else exclude_metadata) or []
    return [m for m in msgs if
            ((mt and m.msg_type in mt) or (md and any(_meta_fns[k](m) for k in md)))
            and not (ex and any(_meta_fns[k](m) for k in ex))]

In [ ]:
ms = await filter_msgs_v1(metadata='exported')
test_is('_8ca984e4' in {m.id for m in ms}, True)

Does this version replicate `filter_msgs_v0`?

In [ ]:
v0 = await filter_msgs_v0(test_msgs)
v1 = await filter_msgs_v1(test_msgs, metadata=['exported', 'echo', 'header'], exclude_metadata='hidden')
test_eq(v0, v1)
v1

[{'id': '_b6147aa6',
  'time_run': '',
  'is_exported': 1,
  'skipped': 0,
  'bookmark': {},
  'i_collapsed': 0,
  'o_collapsed': 0,
  'heading_collapsed': False,
  'i_clamp': False,
  'o_clamp': False,
  'pinned': 0,
  'run': {},
  'content': '##### Header (exported)',
  'output': '',
  'msg_type': 'note',
  'input_tokens': 42,
  'output_tokens': {}},
 {'id': '_fad144f0',
  'time_run': '',
  'is_exported': 0,
  'skipped': 0,
  'bookmark': {},
  'i_collapsed': 0,
  'o_collapsed': 0,
  'heading_collapsed': False,
  'i_clamp': False,
  'o_clamp': False,
  'pinned': 0,
  'run': {},
  'content': '##### Header w/ paragraph (exported)\n\nParagraph.\nLine 2.',
  'output': '',
  'msg_type': 'note',
  'input_tokens': 53,
  'output_tokens': {}},
 {'id': '_2752a761',
  'time_run': '',
  'is_exported': 0,
  'skipped': 0,
  'bookmark': {},
  'i_collapsed': 0,
  'o_collapsed': 0,
  'heading_collapsed': False,
  'i_clamp': False,
  'o_clamp': False,
  'pinned': 0,
  'run': {},
  'content': '##### Not

Let's see more examples and edge cases. (Whenever possible, I much prefer to use `test_*` for testing and, sometimes, it's good to display the result, mainly for my own benefit.)

In [ ]:
ms = await filter_msgs_v1()
test_eq(ms, [])

ms = await filter_msgs_v1(test_msgs, msg_type='note')
test_eq(len(ms), 4)
test_is(all(m.msg_type == 'note' for m in ms), True)

ms = await filter_msgs_v1(test_msgs, msg_type=['note', 'prompt'])
test_eq(len(ms), 6)
test_is(all(m.msg_type in ('note', 'prompt') for m in ms), True)

ms = await filter_msgs_v1(test_msgs, metadata='echo')
test_eq(len(ms), 1)
test_eq(ms[0].id, '_fde18916')

ms = await filter_msgs_v1(test_msgs, metadata='exported', exclude_metadata='hidden')
test_is('_89ee9e7c' in {m.id for m in ms}, False)  # exported+hidden excluded
test_is('_8ca984e4' not in {m.id for m in ms}, True) # exporti but also hidden
test_eq(len(ms), 7)

ms = await filter_msgs_v1(test_msgs, msg_type='note', metadata='exported')
test_is('_cdfea4f6' in {m.id for m in ms}, True)  # non-header note, included via exported
test_is('_b6147aa6' in {m.id for m in ms}, True)  # header note, included via both
test_eq(len(ms), 10)

Next: one of the most important thing `filter_msgs_v1` must do is filter by sections. Thus, we need to add section semantics.

Compose with union, like bare ids list. Precedence: ids -> sections -> metadata. Though seldom agents will use ids and sections in the same call, I think.

The heading is included (but can be filtered afterwards by metadata).

Don't see the point of filtering sections by level (levels don't share semantics) and by ID it's cumbersome for humans (but not for LLMs). Let's start with section name: "## name 1", "### name 2" and simple regex.

In [ ]:
#| export
async def get_section(heading:str, msgs:list[MsgX]=None, dname='') -> list[MsgX]:
    "Return messages in section whose header matches heading"
    if not msgs: return await find_msgs_x(header_section=heading, include_skipped=True, dname=dname)
    lvl = len(re.match(r'^(#+)', heading)[1])
    for i,m in enumerate(msgs):
        if m.msg_type != 'note' or not m.content.startswith(heading): continue
        res = [m]
        for m2 in msgs[i+1:]:
            hs = re.match(r'^(#+)', m2.content) if m2.msg_type == 'note' else None
            if hs and len(hs[1]) <= lvl: break
            res.append(m2)
        return res
    return []

In [ ]:
# exact match
sec = await get_section('##### Header (exported)', test_msgs)
test_eq(len(sec), 1)
test_eq(sec[0].content, '##### Header (exported)')

# section spans until next same-level header
sec = await get_section('## message utils', dlog_msgs)
test_is(sec[0].content.startswith('## message utils'), True)
test_is(all(not (m.msg_type=='note' and re.match(r'^#{1,2}[^#]', m.content)) for m in sec[1:]), True)

# no match returns empty
sec = await get_section('## nonexistent section', dlog_msgs)
test_eq(sec, [])

# subsection doesn't bleed into sibling
sec = await get_section('### message tag', dlog_msgs)
last = sec[-1]
test_is(sec[0].content.startswith('### message tag'), True)
test_is(not any(m.content.startswith('### message metadata') for m in sec), True)

len(sec)

2

In [ ]:
async def filter_msgs_v1(
    msgs:list[Msg]=None,
    ids:str|list=None,
    section:str|list=None,
    msg_type:str|list=None,
    metadata:str|list=None,
    exclude_metadata:str|list=None,
    dname:str='',
) -> list[MsgX]:
    "Filter messages with union semantics; result in dialog order."
    if not any([ids, section, msg_type, metadata]): return []
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    id_set = set([ids] if isinstance(ids, str) else ids or [])
    secs = ([section] if isinstance(section, str) else section) or []
    sec_ids = {m.id for s in secs for m in await get_section(s, msgs)}
    mt = ([msg_type] if isinstance(msg_type, str) else msg_type) or []
    md = ([metadata] if isinstance(metadata, str) else metadata) or []
    ex = ([exclude_metadata] if isinstance(exclude_metadata, str) else exclude_metadata) or []
    return [m for m in msgs if
            (m.id in id_set or m.id in sec_ids or (mt and m.msg_type in mt) or (md and any(_meta_fns[k](m) for k in md)))
            and not (ex and any(_meta_fns[k](m) for k in ex))]

In [ ]:
# ids: single string
ms = await filter_msgs_v1(test_msgs, ids='_b6147aa6')
test_eq(len(ms), 1)
test_eq(ms[0].id, '_b6147aa6')

# ids: list
ms = await filter_msgs_v1(test_msgs, ids=['_b6147aa6', '_4e4b6f65'])
test_eq(len(ms), 2)

# section: single
ms = await filter_msgs_v1(dlog_msgs, section='### message tag')
test_eq(len(ms), 2)
test_is(ms[0].content.startswith('### message tag'), True)

# section: union with msg_type
ms_sec = await filter_msgs_v1(dlog_msgs, section='### message tag')
ms_union = await filter_msgs_v1(dlog_msgs, section='### message tag', msg_type='note')
test_is(len(ms_union) >= len(ms_sec), True)

# section + exclude_metadata
ms = await filter_msgs_v1(dlog_msgs, section='### message metadata & directives', exclude_metadata='hidden')
test_is(all(not is_hidden(m) for m in ms), True)

# ids + section union (no overlap double-counting)
ms = await filter_msgs_v1(dlog_msgs, ids=dlog_msgs[0].id, section='### message tag')
test_is(dlog_msgs[0].id in {m.id for m in ms}, True)
test_is(ms[0].id == dlog_msgs[0].id, True)  # dialog order preserved

# unknown id: silently absent
ms = await filter_msgs_v1(test_msgs, ids='_nonexistent')
test_eq(ms, [])

Seems to work. Next? (We are progressively approaching relational semantics and probably will end up with our own SQL DSL-like lang/algebra, not for now though.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking at what's still missing for practical use, a few natural next steps come to mind:

1. **Range** — `after='_id'` / `before='_id'` or `between=('_id1', '_id2')`: slice a contiguous span by id. Useful for "everything after this header up to that one" without needing `get_section`.

2. **Content search** — `contains='pattern'` (regex): filter by message content. Simple but powerful for finding all cells referencing a symbol, for instance.

3. **`n` / `last`** — `n=10` / `last=5`: take first/last N from the filtered result. Useful for "give me the last 5 exported cells".

Of these, **range** feels most complementary to what's already there and closest to what `get_section` already does implicitly. Content search would open up a new axis entirely. `n`/`last` is more of a post-filter convenience.

Which direction interests you most?

<details class='token-usage-details'><summary>$0.0401</summary>

`total=35,673 | in=35,006 | out=667 | cached=94.4% | cache_new=1,251 | searches=0 | $0.0401`

</details>


`range`, `last` and any operation picking messages by id, I think would be better served by separate helpers whose results can be  passed directly with the `ids` argument to filter additionally by metadata. Don't you think so?

More akin to what we're doing with `filter_msgs_v1` would be introducing "not" semantics with (say) "!" operator. For example, "!exported" would mean "not exported", "!code" -> ['note', 'prompt`, 'raw'], or "!### a section" would mean "everything but section ### a section". Going to DSL step by step.

So the full updated semantics are:

- Parse `!`-prefixed terms from each axis as negations, the rest as positives
- Union all positives across all axes; if none, start from full `msgs`
- Subtract globally all negation matches
- `exclude_metadata` goes away, replaced by e.g. `metadata='!hidden'`

In [ ]:
#| export
def _parse_terms(v):
    terms = ([v] if isinstance(v, str) else v) or []
    return [t for t in terms if not t.startswith('!')], [t[1:] for t in terms if t.startswith('!')]

In [ ]:
async def filter_msgs_v1(
    msgs:list[Msg]=None,
    ids:str|list=None,
    section:str|list=None,
    msg_type:str|list=None,
    metadata:str|list=None,
    dname:str='',
) -> list[MsgX]:
    "Filter messages with union semantics and ! negation; result in dialog order."
    if not any([ids, section, msg_type, metadata]): return []
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    ids_l = ([ids] if isinstance(ids, str) else ids) or []
    sec_pos, sec_neg = _parse_terms(section)
    mt_pos, mt_neg = _parse_terms(msg_type)
    md_pos, md_neg = _parse_terms(metadata)
    pos = set(ids_l)
    for s in sec_pos: pos |= {m.id for m in await get_section(s, msgs)}
    if mt_pos: pos |= {m.id for m in msgs if m.msg_type in mt_pos}
    if md_pos: pos |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_pos)}
    neg = set()
    for s in sec_neg: neg |= {m.id for m in await get_section(s, msgs)}
    if mt_neg: neg |= {m.id for m in msgs if m.msg_type in mt_neg}
    if md_neg: neg |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_neg)}
    base = pos if pos else {m.id for m in msgs}
    return [m for m in msgs if m.id in base - neg]

Tests in SolveIt (fastai style) serve two purposes: correctnes **and** documentation. I think it'd be better to implemente minimal synthetic tests with a minimal set of synthetic (very simple dicts/MsgX) so we can see results without going to other dialog lists elsewhere. For instance, for negation only, we can simply create two minimal MsgX and test visually. And so on. At the end we can go back to the test set and even this full dialog.

In [ ]:
def mk(id, content='',  msg_type='note', **kwargs):
    (res := AD(id=id, msg_type=msg_type, content=content, is_exported=0, output='', skipped=0, pinned=0)).update(**kwargs)
    return msgx(res)

In [ ]:
# negation only: all msgs minus hidden
syn = mk('a', '#| hide', 'code'), mk('b')
ms = await filter_msgs_v1(syn, metadata='!hidden')
test_eq(ms, [syn[1]])

# positive + negation: exported but not hidden
syn = mk('a', '#| export\nx=1', 'code'), mk('b', '#| hide\n#| export\ny=2', 'code'), mk('c', 'just a note')
ms = await filter_msgs_v1(list(syn), metadata=['exported', '!hidden'])
test_eq(ms, [syn[0]])

# msg_type negation: everything but code
syn = mk('a', '## h'), mk('b', 'x=1', 'code'), mk('c', 'q', 'prompt')
ms = await filter_msgs_v1(list(syn), msg_type='!code')
test_eq(ms, [syn[0], syn[2]])

# section negation: all minus a section
syn = mk('a', '## sec1'), mk('b', 'x=1', 'code'), mk('c', '## sec2'), mk('d', 'y=2', 'code')
ms = await filter_msgs_v1(list(syn), section='!## sec1')
test_eq(ms, [syn[2], syn[3]])

# combined: code msgs minus those in a section
ms = await filter_msgs_v1(list(syn), msg_type='code', section='!## sec1')
test_eq(ms, [syn[3]])

# all-negative: full set minus code and hidden
syn = mk('a', '## h'), mk('b', '#| hide\nx=1', 'code'), mk('c', 'y=2', 'code'), mk('d', 'q', 'prompt')
ms = await filter_msgs_v1(list(syn), msg_type='!code', metadata='!hidden')
test_eq(ms, [syn[0], syn[3]])

In [ ]:
v0 = await filter_msgs_v0(test_msgs)
v1 = await filter_msgs_v1(test_msgs, metadata=['exported', 'echo', 'header', '!hidden'])
test_eq(v0, v1)

### content search
Regex filter on message content. Simple, composable, immediately useful for "find all cells referencing symbol foo".

`contains` fits naturally as another axis in `filter_msgs_v1`, with `!` negation like the rest. 

We can use the exact same args than `find_msgs`:
```python
async def find_msgs(
    re_pattern:str='', # Optional regex to search for (re.DOTALL+re.MULTILINE is used)
    use_case:bool=False, # Use case-sensitive matching?
    use_regex:bool=True, # Use regex matching?
```
Just one pattern (with `or` logic if you want multiple matches)

In [ ]:
#| export
def _content_match(m, pat, use_case, use_regex):
    flags = (re.DOTALL|re.MULTILINE)|(0 if use_case else re.IGNORECASE)
    return bool(re.search(pat, m.content, flags) if use_regex else (pat in m.content if use_case else pat.lower() in m.content.lower()))

In [ ]:
async def filter_msgs_v1(
    msgs:list[Msg]=None,
    ids:str|list=None,
    section:str|list=None,
    msg_type:str|list=None,
    metadata:str|list=None,
    re_pattern:str='',
    use_case:bool=False,
    use_regex:bool=True,
    dname:str='',
) -> list[MsgX]:
    "Filter messages with union semantics and ! negation; result in dialog order."
    if not any([ids, section, msg_type, metadata, re_pattern]): return []
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    ids_l = ([ids] if isinstance(ids, str) else ids) or []
    sec_pos, sec_neg = _parse_terms(section)
    mt_pos, mt_neg = _parse_terms(msg_type)
    md_pos, md_neg = _parse_terms(metadata)
    pat_pos, pat_neg = (re_pattern[1:], True) if re_pattern.startswith('!') else (re_pattern, False)
    has_pos = any([ids_l, sec_pos, mt_pos, md_pos, (pat_pos and not pat_neg)])
    pos = set(ids_l)
    for s in sec_pos: pos |= {m.id for m in await get_section(s, msgs)}
    if mt_pos: pos |= {m.id for m in msgs if m.msg_type in mt_pos}
    if md_pos: pos |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_pos)}
    if pat_pos and not pat_neg: pos |= {m.id for m in msgs if _content_match(m, pat_pos, use_case, use_regex)}
    neg = set()
    for s in sec_neg: neg |= {m.id for m in await get_section(s, msgs)}
    if mt_neg: neg |= {m.id for m in msgs if m.msg_type in mt_neg}
    if md_neg: neg |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_neg)}
    if pat_neg: neg |= {m.id for m in msgs if _content_match(m, pat_pos, use_case, use_regex)}
    base = pos if has_pos else {m.id for m in msgs}
    return [m for m in msgs if m.id in base - neg]

Please, minimal tests.

In [ ]:
syn = [mk('a', 'x = foo + 1', 'code'), mk('b', 'y = bar + 2', 'code'), mk('c', 'foo is great', 'note')]

ms = await filter_msgs_v1(list(syn), re_pattern='foo')
test_eq([m.id for m in ms], ['a', 'c'])

ms = await filter_msgs_v1(list(syn), re_pattern='!foo')
test_eq([m.id for m in ms], ['b'])

ms = await filter_msgs_v1(list(syn), re_pattern='FOO', use_case=True)
test_eq(ms, [])

ms = await filter_msgs_v1(list(syn), re_pattern='FOO', use_case=False)
test_eq([m.id for m in ms], ['a', 'c'])

ms = await filter_msgs_v1(list(syn), re_pattern='foo|bar')
test_eq([m.id for m in ms], ['a', 'b', 'c'])

### section map

In [ ]:
#| export
def _header_level(m):
    if m.msg_type != 'note' or not m.content.startswith('#'): return 0
    return len(re.match(r'^(#+)', m.content)[1])


In [ ]:
_header_level(mk('a', '### foo'))

3

In [ ]:
#| export
def section_map(msgs):
    "Return dict mapping full header line -> list of msg ids in that section (inclusive of header)"
    res, cur_hdrs = {}, []
    for m in msgs:
        lvl = _header_level(m)
        if lvl:
            cur_hdrs = [(l,h) for l,h in cur_hdrs if l < lvl]
            hdr = re.match(r'^(#+[^\n]*)', m.content)[1]
            cur_hdrs.append((lvl, hdr))
            for _,h in cur_hdrs: res.setdefault(h, []).append(m.id)
        else:
            for _,h in cur_hdrs: res.setdefault(h, []).append(m.id)
    return res

In [ ]:
def mk_note(id, content): return mk(id, content)
def mk_code(id, content='x=1'): return mk(id, content)

test_eq(section_map([]), {})
test_eq(section_map([mk_code('a'), mk_code('b')]), {})

# messages before first header are not in any section
msgs = [mk_code('pre'),  mk_note('h1', '## Top'), mk_code('c1')]
sm = section_map(msgs)
test_is('pre' not in str(sm), True)
test_eq(sm['## Top'], ['h1', 'c1'])

# nested: child msgs belong to both parent and child sections
msgs = [ mk_note('h1','## Top'),  mk_note('h2','### Sub'), mk_code('c1')]
sm = section_map(msgs)
test_eq(sm['## Top'], ['h1', 'h2', 'c1'])
test_eq(sm['### Sub'], ['h2', 'c1'])

# sibling headers don't bleed into each other
msgs = [ mk_note('h1','## A'), mk_code('c1'),  mk_note('h2','## B'), mk_code('c2')]
sm = section_map(msgs)
test_eq(sm['## A'], ['h1', 'c1'])
test_eq(sm['## B'], ['h2', 'c2'])

# header with no children
test_eq(section_map([ mk_note('h1','## Lonely')]), {'## Lonely': ['h1']})

sm = section_map(await find_msgs(include_meta=True))
sm.keys()

dict_keys(['# dialog', '## message utils', '### message tag', '### message metadata & directives', '### notes disclosure', '### message extended', '### extended `find_msgs`', '## dialog utils', '### filter messages v0', '### digest dialog', '### filter messages v1', '### section map', '### section regex', '### filter messages v2', '### ----', '### import dialog', '### dialog path', '### dialog module', '### dialog export', '### dialog link', '### context usage', '### empty .ipynb', '### message id', '### get output', '### output formatting', '### markdown splitter', '## export -'])

In [ ]:
sm = section_map(dlog_msgs)
list(sm.keys())

['# dialog',
 '## message utils',
 '### message tag',
 '### message metadata & directives',
 '### notes disclosure',
 '### message extended',
 '### extended `find_msgs`',
 '## dialog utils',
 '### filter messages v0',
 '### digest dialog',
 '### filter messages v1',
 '### section map',
 '### section regex',
 '### progressive code disclosure',
 '### filter messages v1 wrap up',
 '### filter messages v2',
 '### ----',
 '### import dialog',
 '### dialog path',
 '### dialog module',
 '### dialog export',
 '### dialog link',
 '### context usage',
 '### empty .ipynb',
 '### message id',
 '### get output',
 '### output formatting',
 '### markdown splitter']

All sections detected correctly.

### section regex

```python
def _scoped_msgs(msgs, ids, section):
    scoped = set()
    if ids is not None: scoped |= ({ids} if isinstance(ids, str) else set(ids))
    if section:
        secs = [section] if isinstance(section, str) else list(section)
        sm = section_map(msgs)
        for s in secs:
            for k,v in sm.items():
                if re.fullmatch(s, k, re.IGNORECASE): scoped |= set(v)
    return [m for m in msgs if m.id in scoped]

async def filter_msgs(msg_type=None, metadata=None, ids=None, section=None, dname='', msgs=None) -> list[Msg]:
    "Filter messages with two-phase semantics: scope (ids, section) then union-filter (msg_type, metadata)"
    if not any([msg_type, metadata, ids is not None, section]): return []
    if msgs is None: msgs = await find_msgs(include_skipped=True, dname=dname)
    if ids is not None or section: msgs = _scoped_msgs(msgs, ids, section)
    if not any([msg_type, metadata]): return msgs
    typs = {msg_type} if isinstance(msg_type, str) else set(msg_type or [])
    metas = {metadata} if isinstance(metadata, str) else set(metadata or [])
    seen, res = set(), []
    for m in msgs:
        if m.id in seen: continue
        if (typs and m.msg_type in typs) or (metas and any(_meta_fns[k](m) for k in metas)):
            res.append(m); seen.add(m.id)
    return res
```

See above an old version of `filter_msgs` that is mostly superseded by out `filter_msgs_v1`. Nevertheless it had a good feature, regex for section titles. Woudl it be hard to add to our version?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Not hard at all. Currently `get_section` matches via `m.content.startswith(heading)` — exact prefix. We'd replace those calls in `filter_msgs_v1` with a `section_map` lookup using `re.fullmatch`, exactly like `_scoped_msgs` does.

The only small design question: `fullmatch` (whole key must match the pattern) or `search` (pattern anywhere in key)? `fullmatch` seems safer to avoid unintended matches.

<details class='token-usage-details'><summary>$0.2859</summary>

`total=50,059 | in=49,744 | out=315 | cached=0.0% | cache_new=48,871 | searches=0 | $0.2859`

</details>


No vey usefull though, maybe later.

### filter messages v1 wrap up

Let's wrap up this section with a modularized version of `filter_msgs_v1` with `scoped_msgs` as discussed above and a new `export_msgs` if necessary.

**modularize.** Extract the scope phase into `_scoped_msgs(msgs, ids, section)` (already sketched in the old `filter_msgs`), leaving `filter_msgs_v1` with just the union/negation filter logic. No behavior change, just cleaner structure.

`_scoped_msgs` first.

The scope phase in `filter_msgs_v1` currently handles `ids` and section pattern resolution (both pos and neg). A clean extraction:

In [ ]:
#| export
def _resolve_sections(msgs, patterns):
    "Return set of msg ids matching any of the section patterns (supports regex via section_map)"
    sm = section_map(msgs)
    ids = set()
    for p in patterns:
        for k, v in sm.items():
            if re.fullmatch(p, k, re.IGNORECASE): ids |= set(v)
    return ids

This replaces the `get_section` calls in `filter_msgs_v1`, and also upgrades section matching to use `section_map` + regex (like the old `filter_msgs`). Then `filter_msgs_v1` calls it for both `sec_pos` and `sec_neg`.

In [ ]:
#| export
async def filter_msgs_v1(
    msgs:list[Msg]=None,
    ids:str|list=None,
    section:str|list=None,
    msg_type:str|list=None,
    metadata:str|list=None,
    re_pattern:str='',
    use_case:bool=False,
    use_regex:bool=True,
    dname:str='',
) -> list[MsgX]:
    "Filter messages with union semantics and ! negation; result in dialog order."
    if not any([ids, section, msg_type, metadata, re_pattern]): return []
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    ids_l = ([ids] if isinstance(ids, str) else ids) or []
    sec_pos, sec_neg = _parse_terms(section)
    mt_pos, mt_neg = _parse_terms(msg_type)
    md_pos, md_neg = _parse_terms(metadata)
    pat_pos, pat_neg = (re_pattern[1:], True) if re_pattern.startswith('!') else (re_pattern, False)
    has_pos = any([ids_l, sec_pos, mt_pos, md_pos, (pat_pos and not pat_neg)])
    pos = set(ids_l)
    pos |= _resolve_sections(msgs, sec_pos)
    if mt_pos: pos |= {m.id for m in msgs if m.msg_type in mt_pos}
    if md_pos: pos |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_pos)}
    if pat_pos and not pat_neg: pos |= {m.id for m in msgs if _content_match(m, pat_pos, use_case, use_regex)}
    neg = set()
    neg |= _resolve_sections(msgs, sec_neg)
    if mt_neg: neg |= {m.id for m in msgs if m.msg_type in mt_neg}
    if md_neg: neg |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_neg)}
    if pat_neg: neg |= {m.id for m in msgs if _content_match(m, pat_pos, use_case, use_regex)}
    base = pos if has_pos else {m.id for m in msgs}
    return [m for m in msgs if m.id in base - neg]

In [ ]:
this_idx = await msg_idx()
msgs = await find_msgs_x()
len(msgs)

231

In [ ]:
v1 = await filter_msgs_v1(msgs[:this_idx], section=['## message utils', '## dialog utils'], metadata=['exported', 'echo', 'header', '!hidden'])
len(v1)

120

It seems a lot of messages. Let's check this is correct. All messages in that list should comply with the constraints: be a header or exported and not hidden, right?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Right. Let's verify both constraints — section membership and the metadata predicate:

In [ ]:
sec_ids = _resolve_sections(msgs[:this_idx], ['## message utils', '## dialog utils'])
if all(is_header(m) or (is_exported(m) and not is_hidden(m)) or is_echo(m) for m in v1) is False:
    print("metadata constraints FAIL")
if all(m.id in sec_ids for m in v1) is False: print('Sections FAIL')

metadata constraints FAIL
Sections FAIL


Some messages in `v1` are not in `sec_ids`. This reveals a design ambiguity: `section` in `filter_msgs_v1` contributes to the *union* (adds section members to the result), but the call was intended to mean "within those sections, return exported/header/echo messages" — i.e. *intersection*.

The old `filter_msgs` handled this with two explicit phases: scope first, then filter. `filter_msgs_v1` doesn't have that distinction.

Two options:
1. Add a `scope` parameter separate from `section` for intersection semantics
2. Keep `section` as union but document it clearly, and use a separate call pattern (pass pre-scoped `msgs`) for intersection

I think the old `filter_msgs` was more intuitive than our version: first scoped the target set, then filter out the result. In fact, I don't understand our version at all, have to spent much more time studying it.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's a fair observation. The two-phase mental model is much clearer:

> *"Give me messages **in** section X (or with ids Y), **that are also** exported/header/echo."*

vs. `filter_msgs_v1`'s union model which reads more like:

> *"Give me messages that are **any of**: in section X, exported, header, echo."*

The union model is powerful for additive queries but breaks down for "filter within a scope" — which is probably the more common real-world need.

Would you like to redesign `filter_msgs_v1` around the two-phase approach, or keep it as-is and introduce a separate function for scoped filtering?

<details class='token-usage-details'><summary>$0.0440</summary>

`total=53,699 | in=53,304 | out=395 | cached=95.9% | cache_new=1,949 | searches=0 | $0.0440`

</details>


### filter messages v2

Let's do a `filter_msgs_v2`. Worth keeping the `v1` for now (need to study it). Try to make it as modular as possible delegating out to external helpers that we already have or new.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Before writing code, let's agree on the shape. Two clean phases:

**Phase 1 — scope** (`_scope_msgs`): if `ids` or `section` given, narrow the working set. If neither given, full `msgs` is the scope.

**Phase 2 — filter** (`_filter_pos_neg`): if `msg_type`/`metadata`/`re_pattern` given, apply union+negation within the scoped set. If none given, return the scope as-is.

Helpers already available: `_resolve_sections`, `_parse_terms`, `_meta_fns`, `_content_match`. The only new ones would be `_scope_msgs` and `_filter_pos_neg`.

Does that decomposition match your mental model?

<details class='token-usage-details'><summary>$0.0375</summary>

`total=54,056 | in=53,570 | out=486 | cached=99.0% | cache_new=270 | searches=0 | $0.0375`

</details>


It does. But your opinion is as important as mine. In fact you have in your weights a lot more info than I could possibly ever have, have seen lot of similar cases and know what's the intended use case of this helpers. What's your take?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

My take: the two-phase design is right, and `filter_msgs_v1`'s union model is worth keeping as a lower-level primitive — it's essentially set algebra on message ids, useful when you want to *compose* result sets. `filter_msgs_v2` is the higher-level, more ergonomic API that most callers will actually use.

One thing I'd flag: `_resolve_sections` is currently `async` but doesn't need to be — `section_map` is sync. Making it sync simplifies the call chain and lets `_scope_msgs` also be sync, so only the top-level `filter_msgs_v2` needs to be async (for the `find_msgs_x` fetch). Worth fixing before we build on it.

<details class='token-usage-details'><summary>$0.0371</summary>

`total=54,321 | in=53,860 | out=461 | cached=99.0% | cache_new=283 | searches=0 | $0.0371`

</details>


Fair enough, go on.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

First, make `_resolve_sections` sync:

Then `_scope_msgs`:

In [ ]:
#| export
def _scope_msgs(msgs, ids, section):
    "Narrow msgs to those matching ids and/or section patterns."
    if ids is None and not section: return msgs
    scoped = set()
    if ids is not None: scoped |= {ids} if isinstance(ids, str) else set(ids)
    if section:
        secs = [section] if isinstance(section, str) else section
        scoped |= _resolve_sections(msgs, secs)
    return [m for m in msgs if m.id in scoped]

`_filter_pos_neg` — applies union+negation within an already-scoped list:

In [ ]:
#| export
def _filter_pos_neg(msgs, msg_type=None, metadata=None, re_pattern='', use_case=False, use_regex=True):
    "Apply union+negation filter to msgs. If no positive terms, return msgs as-is."
    mt_pos, mt_neg = _parse_terms(msg_type)
    md_pos, md_neg = _parse_terms(metadata)
    pat_pos, pat_neg = (re_pattern[1:], True) if re_pattern.startswith('!') else (re_pattern, False)
    has_pos = any([mt_pos, md_pos, pat_pos and not pat_neg])
    if not has_pos and not any([mt_neg, md_neg, pat_neg]): return msgs
    pos = set()
    if mt_pos: pos |= {m.id for m in msgs if m.msg_type in mt_pos}
    if md_pos: pos |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_pos)}
    if pat_pos and not pat_neg: pos |= {m.id for m in msgs if _content_match(m, pat_pos, use_case, use_regex)}
    neg = set()
    if mt_neg: neg |= {m.id for m in msgs if m.msg_type in mt_neg}
    if md_neg: neg |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_neg)}
    if pat_neg: neg |= {m.id for m in msgs if _content_match(m, pat_pos, use_case, use_regex)}
    base = pos if has_pos else {m.id for m in msgs}
    return [m for m in msgs if m.id in base - neg]

As before, I sincerely don't understand `_filter_pos_neg`. I don't say it's wrong, simply I don't get it. Shouldn't each step/filter narrow down the target `msgs` progressively. I see the function iterating every time in the same set of messages and building IMHO an unnecessary set of ids.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The tension is that progressive narrowing gives **AND** semantics ("exported AND header"), but the design calls for **OR** among positives ("exported OR header OR echo"). You can't narrow progressively for OR — you need to collect then union.

That said — is OR actually what you want among positive terms? Or would AND be more natural after scoping?

<details class='token-usage-details'><summary>$0.0410</summary>

`total=55,946 | in=55,519 | out=427 | cached=97.7% | cache_new=1,011 | searches=0 | $0.0410`

</details>


You're right again. `OR` is the correct semantics, I have to refresh my logic. Let's test this.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Agreed, the `v1` signature is cleaner — the visual grouping of scope vs filter args makes the two-phase semantics self-documenting. So `filter_msgs_v2` keeps that exact signature, just changes the internal logic to two-phase (scope then OR-filter) instead of pure union. Ready to write it?

<details class='token-usage-details'><summary>$0.3160</summary>

`total=56,035 | in=55,958 | out=77 | cached=0.0% | cache_new=55,481 | searches=0 | $0.3160`

</details>


In [ ]:
#| export
async def filter_msgs_v2(
    msgs:list[Msg]=None, # List of messages to filter or None to use all messages.
    ids:str|list=None, # Message ids to include (union with section).
    section:str|list=None, # Section patterns to include (union with ids).
    msg_type:str|list=None, # Message types to include.
    metadata:str|list=None, # Metadata keys to to filter on ('exported', 'echo', 'header', 'hidden').
    re_pattern:str='', # Optional regex to search for (re.DOTALL+re.MULTILINE is used).
    use_case:bool=False, # Use case-sensitive matching?
    use_regex:bool=True, # Use regex matching?
    dname:str='', # Dialog to get messages for; defaults to current dialog.
) -> list[MsgX]:
    "Two-phase filter: scope by ids/section, then OR-filter by type/metadata/pattern with ! negation."
    if not any([ids is not None, section, msg_type, metadata, re_pattern]): return []
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    return _filter_pos_neg(_scope_msgs(msgs, ids, section), msg_type, metadata, re_pattern, use_case, use_regex)

### handoff test

Generate a markdown that will serve us as a hand-off to jump start a new discussion easily with the adecuate context. We'll discuss together if that markdown plus additional references and prompts is adecuate for the future task.

Capture: current API surface (`filter_msgs_v2`, `get_section`, `section_map`, `exportmsgs`), the `clamp_msg` design table, the two-axis `note`/`code` levels, the skeleton rules, and pointers to references to check before writing starting a new session.

In [ ]:
v2 = await filter_msgs_v2(msgs[:this_idx], section=['## message utils', '## dialog utils'], metadata=['exported', 'echo', 'header', '!hidden'])
len(v2)

48

In [ ]:
sm = section_map(v2)
list(sm.keys())

['## message utils',
 '### message tag',
 '### message metadata & directives',
 '### notes disclosure',
 '### message extended',
 '### extended `find_msgs`',
 '## dialog utils',
 '### filter messages v0',
 '### digest dialog',
 '### filter messages v1',
 '### section map',
 '### section regex',
 '### filter messages v2']

In [ ]:
fmt = 'md'
md = exportmsgs(v2, hdr_clamp='header++', fmt=fmt,
    frontmatter=dict(path=find_dname(), description='filter messages sections of pote.dutil dialog', n_msgs=len(msgs), fmt=fmt))
# print(md)
await link_msg(md);

---
digest: solveit-dialog
path: /prj/pote/nbs/00_dialog
exported_at: 2026-04-24T08:56:06+00:00
description: 'filter messages sections of pote.dutil dialog'
stats: {total: 231, exported: 48}
conventions: '<!-- #<msg_id> <msg_type> --> precedes each block; ```output blocks contain cell output; ```response blocks contain model response'
---

<!-- #_279f4bb3 note -->
## message utils
Core primitives for working with solveit messages: a `Msg` type alias, utilities for reading nbdev directives and message metadata, progressive disclosure of note content, and an extended `MsgX` subclass with convenience properties.

<!-- #_e229f022 note -->
### message tag
Defines `Msg` as a type alias for `adict`, providing a concise, readable type annotation for solveit message objects throughout the dialog.

<!-- #_68d5b75e code -->
```python
Msg: TypeAlias = AD
```

<!-- #_4b91a31a note -->
### message metadata & directives
Utilities for inspecting solveit message metadata. `is_exported` checks if a message is marked for export (via solveit metadata or nbdev `#|export`/`#|exports`/`#|exporti` directives). `is_hidden` checks if a message is hidden from doc generation via `#|hide`. `is_echo` checks if a code message is marked to show source via `#|echo`. `is_header` checks if the note message start with a markdown heading.

<!-- #_8ca984e4 code -->
```python
_exports = {'exporti', 'exports', 'export'}
_hidden = {'hide'}
_echo = {'echo', 'echo:'}
```

<!-- #_4426db27 code -->
```python
def is_exported(m:Msg) -> bool:
    "Is message exported with solveit metadata or nbdev directive?"
    return m.is_exported or (m.msg_type == 'code' and bool(
        extract_directives(AD(source=m.content), remove=False).keys() & _exports))

def is_hidden(m:Msg) -> bool:
    "Is message hidden by a nbdev directive (only for doc generation, not AI visibility)?"
    return m.msg_type == 'code' and bool(
        extract_directives(AD(source=m.content), remove=False).keys() & _hidden)

def is_echo(m:Msg) -> bool:
    "Does message have an echo directive that isn't false?"
    if m.msg_type != 'code': return False
    d = {k:v for k,v in extract_directives(AD(source=m.content), remove=False).items() if k in _echo}
    return bool(d) and all('false' not in v for v in d.values())

def is_header(m:Msg) -> bool:
    "Does note message content start by a markdown header?"
    return m.msg_type == 'note' and m.content.startswith('#')
```

<!-- #_c0c7efd9 code -->
```python
test_is(is_hidden(Msg(msg_type='code', content='#|hide\n\nsome content')), True)

test_is(is_header(Msg(msg_type='note', content='# a')), True)
test_is(is_header(Msg(msg_type='note', content=' ### a')), False)

test_is(is_echo(Msg(msg_type='code', content='#| echo\nprint(1)')), True)
test_is(is_echo(Msg(msg_type='code', content='#|hide\n#| echo: true\nprint(1)')), True)
test_is(is_echo(Msg(msg_type='code', content='#| echo: false\nprint(1)')), False)
test_is(is_echo(Msg(msg_type='note', content='#| echo\nsome text')), False)
test_is(is_echo(Msg(msg_type='code', content='print(1)')), False)
```

<!-- #_5759b85e note -->
### notes disclosure
`clamp_note` supports four levels of **progressive disclosure** for markdown notes.
| Level | Returns |
|---|---|
| `'header'` | The first line, if it is a markdown header; otherwise `''` |
| `'header+'` | Header + first non-blank line after it |
| `'header++'` | Header + first paragraph, only if it immediately follows the header |
| `int` (e.g. `30`) | First *n* lines, regardless of structure |
All levels return `''` if the content is empty or (for header levels) has no leading header.

<!-- #_f463651b code -->
```python
def clamp_note(
    c:str, # Markdown note content
    level:str|int='header' # Disclosure level: 'header', 'header+', 'header++', or int for max lines
) -> str:
    "Return a progressively disclosed excerpt of a markdown note."
    c = c.strip().splitlines()
    if c:
        if isinstance(level, int): return '\n'.join(c[:level])
        if level == 'header': return c[0] if c[0][0] == '#' else ''
        if level == 'header+':
            i, res = 0, c[0]
            if res[0] != '#': return ''
            while (i:=i+1) < len(c):
                if c[i]: return f"{res}\n{c[i]}"
            return res
        if level.startswith('header++'):
            header = c[0] if c[0][0] == '#' else ''
            if not header: return ''
            blocks = [b.strip() for b in '\n'.join(c[1:]).split('\n\n') if b.strip()]
            if blocks:
                first = blocks[0].splitlines()[0]
                if not first.startswith('#') and not first.startswith(('```', '~~~')):
                    return f"{header}\n{blocks[0]}"
            return header
    return ''
```

<!-- #_d1bb1efe code -->
```python
cnt = '#### test\n> Paragraph 1.\nLine.\n\nParagraph 2\n```python\na=13\n```'
test_eq(clamp_note(cnt, 'header'), '#### test')

test_eq(clamp_note(cnt, 'header+'), '#### test\n> Paragraph 1.')
test_eq(clamp_note(cnt, 'header++'), '#### test\n> Paragraph 1.\nLine.')
```

<!-- #_df3f5a47 note -->
### message extended
Extends the base `Msg` (`adict`) type with `MsgX`, a subclass that adds convenience properties for inspecting message metadata.
`msgx()` casts any `Msg` in-place to `MsgX` (zero-copy, via `__class__` reassignment). `MsgX` then exposes `.exported_x` and `.hidden_x` as properties wrapping `is_exported`/`is_hidden`.

<!-- #_4a03c72d code -->
```python
class MsgX(Msg): ...
```

<!-- #_8347e9f5 code -->
```python
def msgx(msg:adict) -> None:
    "Return `msg` cast to MsgX."
    msg.__class__ = MsgX
    return msg
```

<!-- #_88f12bd3 code -->
```python
@patch(as_prop=True)
def exported_x(self:MsgX) -> bool: return is_exported(self)
@patch(as_prop=True)
def hidden_x(self:MsgX) -> bool: return is_hidden(self)
@patch(as_prop=True)
def echo_x(self:MsgX) -> bool: return is_echo(self)
@patch(as_prop=True)
def header_x(self:MsgX) -> bool: return is_header(self)
```

<!-- #_1dc1a024 note -->
### extended `find_msgs`
`find_msgs` returning MsgX.

<!-- #_02d7356e code -->
```python
@delegates(find_msgs)
async def find_msgs_x(*args, **kwargs) -> list[MsgX]:
    "find_msgs returning MsgX instances."
    ms = await find_msgs(*args, **kwargs)
    for m in ms: m.__class__ = MsgX
    return ms
```

<!-- #_e7e915df note -->
## dialog utils
Utilities for treating dialogs as first-class objects: reading and writing message content across sessions, resolving their backing files and Python modules, and introspecting runtime state such as token usage and cell outputs.

<!-- #_239b8e2a note -->
### filter messages v0
Filters a list of messages down to those relevant for export or documentation: exported (non-hidden) code cells, note headers, and cells with an echo directive.

<!-- #_32c810d5 code -->
```python
async def filter_msgs_v0(
    msgs:list[Msg]=None, # List of messages to filter or None to use all messages.
    dname:str='', # Dialog to get messages for; defaults to current dialog.
) -> list[MsgX]:
    "Return notes, exported messages not hidden and code cells that have echo directive."
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    return [m for m in msgs if (is_exported(m) and not is_hidden(m)) or is_header(m) or is_echo(m)]
```

<!-- #_69a9adb3 note -->
### digest dialog
Format a filtered digest of dialog messages for AI consumption. Supports `xml` (preferred), `md`, and `json` formats. Headers are progressively disclosed via `clamp_note` to keep digests compact. YAML frontmatter is prepended via `mk_frontmatter`.

<!-- #_643a905d code -->
```python
_pmark = '**🤖Reply🤖**<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->\n'
```

<!-- #_d2685927 code -->
```python
def _mk_tag(m, mtyp, ids):
    # parts = ([f'#{m.id}'] if ids else []) + [mtyp] + (['echo'] if echo else []) + (['skipped'] if m.skipped else []) + (['pinned'] if m.pinned else [])
    parts = ([f'#{m.id}'] if ids else []) + [mtyp]
    return ' '.join(parts)
```

<!-- #_d2bc045a code -->
```python
def output2md(c, langid='output'):
    return f'''```{langid}\n{c}\n```''' if c else ''
```

<!-- #_ff4409be code -->
```python
def strip_directives(c): return ('\n'.join(l for l in c.splitlines() if not l.startswith('#|'))).strip()
```

<!-- #_9ba03aa3 code -->
```python
_striptag_pats = (
    re.compile(r'^#\|\W*\w+: \w+\W*$\n?', re.MULTILINE),
    re.compile(r'^<!-- \w+: \w+ -->\s*\n?', re.MULTILINE))

def strip_tags(c):
    for pat in _striptag_pats: c = pat.sub('', c)
    return c.strip()
```

<!-- #_24b7596b code -->
```python
def _xml_msg(m:MsgX, ids, output, hdr_clamp):
    mtyp = m.msg_type
    c, attrs = '', (f' id="{m.id}"' if ids else '') + f' type="{mtyp}"'
    match mtyp:
        case 'note':
            clamp = not m.exported_x and m.header_x
            c = escape(strip_tags(clamp_note(m.content, hdr_clamp) if clamp else m.content))
            if c: c = f'<content>{c}</content>'
        case 'code':
            src = escape(strip_directives(m.content))
            out = f'\n  <output>{escape(m.output)}</output>' if output and m.output else ''
            if src or out: c = f'\n  <source>{src}</source>{out}\n'
        case 'prompt':
            p = escape(strip_tags(strip_directives(m.content)))
            r = f'\n  <response>{escape(m.output)}</response>' if m.output else ''
            if p or r: c = f'<prompt>{p}</prompt>{r}'
        case 'raw':
            c = escape(strip_tags(strip_directives(m.content)))
            if c: c = f'<msg{attrs}><content>{c}</content></msg>'
    return f'<msg{attrs}>{c}</msg>' if c else ''
```

<!-- #_d183b554 code -->
```python
def _md_msg(m, ids, output, hdr_clamp):
    mtyp, c = m.msg_type, None
    match mtyp:
        case 'note':
            clamp = not m.exported_x and m.header_x
            c = strip_tags(clamp_note(m.content, hdr_clamp) if clamp else m.content)
        case 'code':
            c = f'```python\n{strip_directives(m.content)}\n```'
            if output and m.output: c += f'\n{output2md(m.output)}'
        case 'prompt':
            c = strip_tags(strip_directives(m.content))
            if output and m.output: c += f'\n{output2md(m.output, 'response')}'
        case 'raw': c = strip_tags(strip_directives(m.content))
    return f'<!-- {_mk_tag(m, mtyp, ids)} -->\n{c}\n' if c else ''
```

<!-- #_39d5c5d5 code -->
```python
def _json_msg(m, expd, echo, ids, output, hdr_clamp):
    mtyp = m.msg_type
    d = dict(type=mtyp)
    if ids: d['id'] = m.id
    match mtyp:
        case 'note':
            clamp = not m.exported_x and m.header_x
            d['content'] = strip_tags(clamp_note(m.content, hdr_clamp) if clamp else m.content)
        case 'code':
            d['source'] = strip_directives(m.content)
            if output and m.output: d['output'] = m.output
        case 'prompt':
            d['prompt'] = strip_tags(strip_directives(m.content))
            d['response'] = m.output
        case 'raw': d['content'] = strip_tags(strip_directives(m.content))
    return d
```

<!-- #_1fe5afdb code -->
```python
_convs = dict(
    xml="msg elements have id and type attributes; source/output are child elements of code msgs; prompt/response are child elements of prompt msgs",
    json="msgs is a list of objects with type and id fields; code objects have source and output fields; prompt objects have prompt and response fields; others have content",
    md="<!-- #<msg_id> <msg_type> --> precedes each block; ```output blocks contain cell output; ```response blocks contain model response")
```

<!-- #_c02cd206 code -->
```python
def mk_frontmatter(path='', description='', n_msgs=0, n_exported=0, fmt='xml'):
    from datetime import datetime, timezone
    return ("---\n"
"digest: solveit-dialog\n"
f"path: {path}\n"
f"exported_at: {datetime.now(timezone.utc).isoformat(timespec='seconds')}\n"
f"description: {description!r}\n"
f"stats: {{total: {n_msgs}, exported: {n_exported}}}\n"
f"conventions: '{_convs[fmt]}'\n"
"---\n"
)
```

<!-- #_37710188 code -->
```python
def exportmsgs(
    msgs:list[Msg],
    hdr_clamp='header+',  # header level
    output=True,  # include code messages output
    ids=True,  # include message ids in cell comments
    frontmatter:dict=None,  # if present, call `mk_frontmatter` with this dict as args
    fmt='xml',  # 'xml'|'md'|'json'
):
    "Export messages as a digest for AI consumption"
    fn = {'xml': _xml_msg, 'md': _md_msg, 'json': _json_msg}[fmt]
    parts = [p for m in msgs if (p := fn(m, ids, output, hdr_clamp))]
    if fmt == 'json': res = json.dumps({'msgs': parts}, indent=2)
    else:
        body = '\n'.join(parts)
        res = f'<digest>\n{body}\n</digest>' if fmt=='xml' else body
    if frontmatter:
        frontmatter.update(n_exported=len(parts))
        fm = mk_frontmatter(**frontmatter)
    return f"{fm}\n{res}" if frontmatter else res
```

<!-- #_1b1594d4 note -->
Format a filtered, clamped digest of dialog messages for AI consumption in other sessions. Three output formats are supported: `xml` (preferred — unambiguous structure), `md`, and `json`, each implemented by a parallel `_xml_msg`/`_md_msg`/`_json_msg` function dispatched by `exportmsgs`.

**Clamping:** note content is progressively disclosed via `clamp_note(level)` — from header-only to header+paragraph — unless the note is fully exported. This keeps digests compact while preserving section structure and intent.

**Frontmatter:** `mk_frontmatter` prepends YAML metadata (path, timestamp, stats, format conventions) so the consuming LLM knows what it's looking at.

<!-- #_3fc335de note -->
### filter messages v1

<!-- #_dde7bb2c prompt -->
`find_msgs` is a staple of SolveIt, a very powerful introspection tool. But it have some limitations (see above how it doesn't honor explicit `nbdev` export directives) and I need more search granularity (e.g., to improve `exportmsgs`).

We're going to implement `filter_msgs_v1`, a wrapper over `find_msgs_x` with progressively improved search semantics. With no arguments, `filter_msgs_v1` return an empty list.

```python
def filter_msgs_v1(*args, **kwargs) -> list[MsgX]: ...
```

Let's start with `nbdev` export directives: if given `exported=True`, `filter_msgs_v1` should return **all** exported messages, those marked with SolveIt UI toggle **and** the ones with `nbdev` exported directives.
```response
<output result="pending" reason="incomplete"/>
```

<!-- #_7800f21f code -->
```python
_meta_fns = dict(exported=is_exported, echo=is_echo, header=is_header, hidden=is_hidden)
```

<!-- #_187cb632 code -->
```python
async def get_section(heading:str, msgs:list[MsgX]=None, dname='') -> list[MsgX]:
    "Return messages in section whose header matches heading"
    if not msgs: return await find_msgs_x(header_section=heading, include_skipped=True, dname=dname)
    lvl = len(re.match(r'^(#+)', heading)[1])
    for i,m in enumerate(msgs):
        if m.msg_type != 'note' or not m.content.startswith(heading): continue
        res = [m]
        for m2 in msgs[i+1:]:
            hs = re.match(r'^(#+)', m2.content) if m2.msg_type == 'note' else None
            if hs and len(hs[1]) <= lvl: break
            res.append(m2)
        return res
    return []
```

<!-- #_c1f52e60 code -->
```python
def _parse_terms(v):
    terms = ([v] if isinstance(v, str) else v) or []
    return [t for t in terms if not t.startswith('!')], [t[1:] for t in terms if t.startswith('!')]
```

<!-- #_67824193 code -->
```python
def _content_match(m, pat, use_case, use_regex):
    flags = (re.DOTALL|re.MULTILINE)|(0 if use_case else re.IGNORECASE)
    return bool(re.search(pat, m.content, flags) if use_regex else (pat in m.content if use_case else pat.lower() in m.content.lower()))
```

<!-- #_963c14a0 note -->
### section map

<!-- #_00ff0767 code -->
```python
def _header_level(m):
    if m.msg_type != 'note' or not m.content.startswith('#'): return 0
    return len(re.match(r'^(#+)', m.content)[1])
```

<!-- #_56979fd2 code -->
```python
def section_map(msgs):
    "Return dict mapping full header line -> list of msg ids in that section (inclusive of header)"
    res, cur_hdrs = {}, []
    for m in msgs:
        lvl = _header_level(m)
        if lvl:
            cur_hdrs = [(l,h) for l,h in cur_hdrs if l < lvl]
            hdr = re.match(r'^(#+[^\n]*)', m.content)[1]
            cur_hdrs.append((lvl, hdr))
            for _,h in cur_hdrs: res.setdefault(h, []).append(m.id)
        else:
            for _,h in cur_hdrs: res.setdefault(h, []).append(m.id)
    return res
```

<!-- #_dfc988f1 note -->
### section regex

<!-- #_16a9f89d code -->
```python
def _resolve_sections(msgs, patterns):
    "Return set of msg ids matching any of the section patterns (supports regex via section_map)"
    sm = section_map(msgs)
    ids = set()
    for p in patterns:
        for k, v in sm.items():
            if re.fullmatch(p, k, re.IGNORECASE): ids |= set(v)
    return ids
```

<!-- #_073fd07f code -->
```python
async def filter_msgs_v1(
    msgs:list[Msg]=None,
    ids:str|list=None,
    section:str|list=None,
    msg_type:str|list=None,
    metadata:str|list=None,
    re_pattern:str='',
    use_case:bool=False,
    use_regex:bool=True,
    dname:str='',
) -> list[MsgX]:
    "Filter messages with union semantics and ! negation; result in dialog order."
    if not any([ids, section, msg_type, metadata, re_pattern]): return []
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    ids_l = ([ids] if isinstance(ids, str) else ids) or []
    sec_pos, sec_neg = _parse_terms(section)
    mt_pos, mt_neg = _parse_terms(msg_type)
    md_pos, md_neg = _parse_terms(metadata)
    pat_pos, pat_neg = (re_pattern[1:], True) if re_pattern.startswith('!') else (re_pattern, False)
    has_pos = any([ids_l, sec_pos, mt_pos, md_pos, (pat_pos and not pat_neg)])
    pos = set(ids_l)
    pos |= _resolve_sections(msgs, sec_pos)
    if mt_pos: pos |= {m.id for m in msgs if m.msg_type in mt_pos}
    if md_pos: pos |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_pos)}
    if pat_pos and not pat_neg: pos |= {m.id for m in msgs if _content_match(m, pat_pos, use_case, use_regex)}
    neg = set()
    neg |= _resolve_sections(msgs, sec_neg)
    if mt_neg: neg |= {m.id for m in msgs if m.msg_type in mt_neg}
    if md_neg: neg |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_neg)}
    if pat_neg: neg |= {m.id for m in msgs if _content_match(m, pat_pos, use_case, use_regex)}
    base = pos if has_pos else {m.id for m in msgs}
    return [m for m in msgs if m.id in base - neg]
```

<!-- #_bb1f6ca6 note -->
### filter messages v2

<!-- #_9899a2d5 code -->
```python
def _scope_msgs(msgs, ids, section):
    "Narrow msgs to those matching ids and/or section patterns."
    if ids is None and not section: return msgs
    scoped = set()
    if ids is not None: scoped |= {ids} if isinstance(ids, str) else set(ids)
    if section:
        secs = [section] if isinstance(section, str) else section
        scoped |= _resolve_sections(msgs, secs)
    return [m for m in msgs if m.id in scoped]
```

<!-- #_db9f095e code -->
```python
def _filter_pos_neg(msgs, msg_type=None, metadata=None, re_pattern='', use_case=False, use_regex=True):
    "Apply union+negation filter to msgs. If no positive terms, return msgs as-is."
    mt_pos, mt_neg = _parse_terms(msg_type)
    md_pos, md_neg = _parse_terms(metadata)
    pat_pos, pat_neg = (re_pattern[1:], True) if re_pattern.startswith('!') else (re_pattern, False)
    has_pos = any([mt_pos, md_pos, pat_pos and not pat_neg])
    if not has_pos and not any([mt_neg, md_neg, pat_neg]): return msgs
    pos = set()
    if mt_pos: pos |= {m.id for m in msgs if m.msg_type in mt_pos}
    if md_pos: pos |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_pos)}
    if pat_pos and not pat_neg: pos |= {m.id for m in msgs if _content_match(m, pat_pos, use_case, use_regex)}
    neg = set()
    if mt_neg: neg |= {m.id for m in msgs if m.msg_type in mt_neg}
    if md_neg: neg |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_neg)}
    if pat_neg: neg |= {m.id for m in msgs if _content_match(m, pat_pos, use_case, use_regex)}
    base = pos if has_pos else {m.id for m in msgs}
    return [m for m in msgs if m.id in base - neg]
```

<!-- #_ad25c0ee code -->
```python
async def filter_msgs_v2(
    msgs:list[Msg]=None, # List of messages to filter or None to use all messages.
    ids:str|list=None, # Message ids to include (union with section).
    section:str|list=None, # Section patterns to include (union with ids).
    msg_type:str|list=None, # Message types to include.
    metadata:str|list=None, # Metadata keys to to filter on ('exported', 'echo', 'header', 'hidden').
    re_pattern:str='', # Optional regex to search for (re.DOTALL+re.MULTILINE is used).
    use_case:bool=False, # Use case-sensitive matching?
    use_regex:bool=True, # Use regex matching?
    dname:str='', # Dialog to get messages for; defaults to current dialog.
) -> list[MsgX]:
    "Two-phase filter: scope by ids/section, then OR-filter by type/metadata/pattern with ! negation."
    if not any([ids is not None, section, msg_type, metadata, re_pattern]): return []
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    return _filter_pos_neg(_scope_msgs(msgs, ids, section), msg_type, metadata, re_pattern, use_case, use_regex)
```

<!-- linkedto: _9cf954a5 -->

Before going on, I know is difficult, you can't forget what's in your context, but try. Take a look a `md` result above. Is a good starting point for the future discussion? Does fullfil the goal of geneerating "good" context?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking at it critically, it's good but incomplete for the stated goal. A few observations:

**What works well:** the `header++` clamping gives just enough context per section without dumping everything; the frontmatter orients a new session; the full exported code is there and runnable.

**What's missing:**
- `filter_msgs_v1`, `filter_msgs_v2`, `_scope_msgs`, `_resolve_sections`, `_filter_pos_neg` — none exported yet, so they don't appear. A fresh session would have no idea what was built.
- The key *design decisions* are absent: `!` negation semantics, union vs two-phase debate, the `has_pos` fix. Those live only in the prompt/response messages which were excluded.
- `mk`, `_parse_terms`, `_content_match`, `section_map`, `_header_level` — same problem.

**Suggestion:** before generating the handoff, mark the new functions as exported, then also include a short handwritten note message summarising the design decisions that are invisible in code alone. The md export can carry the code; the note carries the *why*.

<details class='token-usage-details'><summary>$0.0744</summary>

`total=64,644 | in=64,007 | out=637 | cached=90.5% | cache_new=5,853 | searches=0 | $0.0744`

</details>


Of course, it was a test. I've got to sweep the whole dialog cleaning and marking messages. That isn't the final result at all.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Exactly — the handoff quality is only as good as the curation upstream. The export is the easy part; the real work is the sweep: marking exports, writing good section notes, and deciding what the next session needs to know vs. what it can infer from code alone.

When you're ready to do that sweep, `filter_msgs_v2` with `section` + `metadata` is now a good tool for it — you can pull any section, inspect what's exported, and patch gaps.

<details class='token-usage-details'><summary>$0.0343</summary>

`total=64,471 | in=64,360 | out=111 | cached=99.1% | cache_new=378 | searches=0 | $0.0343`

</details>


The two-phase design works well — 33 messages, cleanly scoped and filtered. Now let's draft the handoff markdown:

```python
handoff_msgs = await filter_msgs_v2(msgs[:this_idx], 
    section=['## message utils', '## dialog utils'],
    metadata=['exported', 'echo', 'header', '!hidden'])

fmt = 'md'
handoff = exportmsgs(handoff_msgs, hdr_clamp='header++', fmt=fmt, frontmatter=dict(
    path=find_dname(), 
    description='Handoff: filter_msgs_v1/v2 and section filtering in pote.dutil',
    n_msgs=len(msgs), fmt=fmt))

Path('data/handoff_filter_msgs.md').write_text(handoff)
print(f"{len(handoff)} chars, {len(handoff_msgs)} msgs")
```

## token budget

Let's think about the content budget. That would be very good to have but I don't see any way of making it deterministic. I don't want to lose whole messages or truncate content, so we would have to rely on LLM compaction, always tricky and dangerous. Ideas, suggestions?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The constraint "no lost messages, no truncated content" is very tight. A few angles that are deterministic:

1. **Progressive note disclosure** — already in `clamp_note`. Dialing down from `header++` to `header+` to `header` is lossless (full content still in dialog) and can shave tokens significantly for long notes.

2. **Output stripping** — code cell outputs are regenerable, not "content" in the same sense. Dropping or truncating outputs is safe and often the biggest win.

3. **Metadata-only mode** — return just ids + types + first line of content as a table of contents, letting the consumer request full sections on demand. But this requires a back-and-forth protocol.

Which of these feels closest to what you need?

<details class='token-usage-details'><summary>$0.0363</summary>

`total=50,548 | in=50,000 | out=548 | cached=99.4% | cache_new=283 | searches=0 | $0.0363`

</details>


## Code message disclosure
Progressive disclosure of Python source code at three levels: signatures only (level 1), signatures with docstrings and comments (level 2), and full source (level 3). Per-cell, preserving message boundaries and ids.

See <a href="/dialog_?name=prj%2Fpote%2Fexplorer%2Fcode_disclosure%2Fcode_disclosure_v1" target="_blank">prj/pote/explorer/code_disclosure/code_disclosure_v2</a>

In [ ]:
def count_tokens(src, model='cl100k_base'):
    "Count tokens in `src` using `model` encoding"
    return len(tiktoken.get_encoding(model).encode(src))

In [ ]:
count_tokens("def foo(x:int) -> bool: return x > 0")

13

In [ ]:
#| export
def _is_docstring(n): return isinstance(n, ast.Expr) and isinstance(n.value, ast.Constant) and isinstance(n.value.value, str)

def _reindent(src, n): return '\n'.join(' '*n + l for l in src.splitlines())

def _postprocess(s):
    "Collapse triple+ blank lines and hoist lone `...` body onto the signature line"
    s = re.sub(r'\n{3,}', '\n\n', s)
    return re.sub(r':\n[ \t]+\.\.\.', ': ...', s)

_compound_types = (ast.If, ast.For, ast.While, ast.With, ast.Try, ast.Match)

In [ ]:
#| export
def _strip_private(tree):
    "Remove private (underscore-prefixed) top-level nodes and class methods from `tree` in-place"
    is_priv = lambda n: getattr(n, 'name', '').startswith('_')
    tree.body = [n for n in tree.body if not is_priv(n)]
    for node in tree.body:
        if isinstance(node, ast.ClassDef):
            node.body = [n for n in node.body if not is_priv(n)]
    return tree

In [ ]:
#| export
def stub_fn(atok, fn, level=1):
    "Return stubbed source for function `fn` at disclosure `level`"
    if level == 3: return atok.get_text(fn)
    start_line = fn.decorator_list[0].lineno if fn.decorator_list else fn.lineno
    body_start = fn.body[0].lineno - start_line
    if level == 1 or body_start == 0:
        n = copy.deepcopy(fn)
        n.body = [ast.Expr(ast.Constant(...))]
        return _reindent(ast.unparse(n), fn.col_offset)
    indent = ' ' * fn.body[0].col_offset
    lines = atok.get_text(fn).splitlines()
    if _is_docstring(fn.body[0]):
        doc_end = fn.body[0].end_lineno - start_line + 1
        return '\n'.join(lines[:body_start] + lines[body_start:doc_end] + [f'{indent}...'])
    return '\n'.join(lines[:body_start] + [f'{indent}...'])

In [ ]:
_src = 'def f(a:int # arg\n) -> int: # ret\n    "a doc"\n    return a+20'
_atok = asttokens.ASTTokens(_src, parse=True)
_fn = _atok.tree.body[0]
test_eq(stub_fn(_atok, _fn, 1), 'def f(a: int) -> int:\n    ...')
test_eq(stub_fn(_atok, _fn, 2), 'def f(a:int # arg\n) -> int: # ret\n    "a doc"\n    ...')
test_eq(stub_fn(_atok, _fn, 3), _src)

In [ ]:
#| export
def _class_chunk(atok, node, level):
    "Return disclosed source for class `node` at `level`"
    if level == 3: return atok.get_text(node)
    start_line = node.decorator_list[0].lineno if node.decorator_list else node.lineno
    body_start = node.body[0].lineno - start_line
    lines = atok.get_text(node).splitlines()
    hdr = '\n'.join(lines[:body_start])
    body = []
    for i,n in enumerate(node.body):
        if isinstance(n, functypes): body.append(stub_fn(atok, n, level))
        elif isinstance(n, ast.Expr):
            if i == 0 and _is_docstring(n) and level >= 2: body.append(' '*n.col_offset + atok.get_text(n))
        else: body.append(' '*n.col_offset + atok.get_text(n))
    return hdr + '\n' + '\n'.join(body)

def _compound_chunk(atok, node, level):
    "Return disclosed source for compound statement `node` at `level`"
    if level == 3: return atok.get_text(node)
    text = atok.get_text(node)
    fns = sorted([n for n in ast.walk(node) if n is not node and isinstance(n, (*functypes, ast.ClassDef))],
                 key=lambda n: (n.lineno, n.col_offset), reverse=True)
    for n in fns:
        stub = stub_fn(atok, n, level) if isinstance(n, functypes) else _class_chunk(atok, n, level)
        lines = stub.splitlines()
        if lines[0].startswith(' ' * n.col_offset): lines[0] = lines[0][n.col_offset:]
        text = text.replace(atok.get_text(n), '\n'.join(lines), 1)
    return text

In [ ]:
#| export
def disclose(src:str, level=1, strip_private=False) -> str:
    "Disclose Python source at `level` (1=sigs, 2=sigs+docs, 3=full)"
    if not src.strip(): return ''
    try:
        atok = asttokens.ASTTokens(src, parse=True)
        if strip_private: _strip_private(atok.tree)
        chunks = []
        for i,node in enumerate(atok.tree.body):
            if isinstance(node, functypes): chunks.append(stub_fn(atok, node, level))
            elif isinstance(node, ast.ClassDef): chunks.append(_class_chunk(atok, node, level))
            elif isinstance(node, _compound_types): chunks.append(_compound_chunk(atok, node, level))
            elif isinstance(node, ast.Expr) and not (i == 0 and _is_docstring(node) and level >= 2): pass
            elif isinstance(node, (ast.Import, ast.ImportFrom)) and level == 1: pass
            else: chunks.append(atok.get_text(node))
        return _postprocess('\n\n'.join(chunks))
    except SyntaxError as e:
        warnings.warn(f'Syntax error in source, returning raw: {e}')
        return src

In [ ]:
test_eq(disclose('', 1), '')
test_eq(disclose("def bar(): pass", 1), "def bar(): ...")
test_eq(disclose("def bar(): pass", 2), "def bar(): ...")
test_eq(disclose("def bar(): pass", 3), "def bar(): pass")

_s = 'def f(a:int # arg\n) -> int: # ret\n    "a doc"\n    return a+20'
test_eq(disclose(_s, 1), "def f(a: int) -> int: ...")
test_eq(disclose(_s, 2), 'def f(a:int # arg\n) -> int: # ret\n    "a doc"\n    ...')
test_eq(disclose(_s, 3), _s)

### dialog and notebook disclosure
Integrates per-cell `disclose` with solveit dialog and notebook file workflows. Three inclusion tiers: `'exported'` (module API), `'echo'` (API + doc examples), `'all'`.

In [ ]:
#| export
_include_filter = dict(
    exported=lambda m: is_exported(m),
    echo=lambda m: is_exported(m) or is_echo(m),
    all=lambda m: True)

def _compact(s): return re.sub(r'\n{2,}', '\n', s).strip()

def _xml_cell(content, **attrs):
    attr_str = ''.join(f' {k}="{v}"' for k,v in attrs.items())
    return f'<cell{attr_str}>{content}</cell>'

In [ ]:
#| export
async def dialog_code(dname='', include='all'):
    "Return list of (id, source) tuples for code cells filtered by `include`"
    msgs = await find_msgs_x(dname=dname, msg_type='code', include_output=False)
    return [(m.id, m.content) for m in msgs if _include_filter[include](m)]

In [ ]:
#| export
def _disclose_srcs(srcs, level=1, strip_private=False, fmt='text', dname=''):
    "Disclose list of (id, source) tuples at `level`"
    chunks = [(mid, disclose(src, level, strip_private)) for mid,src in srcs]
    if fmt == 'xml':
        cells = '\n'.join(_xml_cell(escape(src), id=mid) for mid,src in chunks)
        return f'<dialog level="{level}" path="{dname or "."}">\n{cells}\n</dialog>'
    if fmt == 'json':
        cells = [dict(id=mid, source=src) for mid,src in chunks]
        return json.dumps(dict(level=level, path=dname or '.', cells=cells), indent=2)
    return _compact('\n\n'.join(src for _,src in chunks))

async def disclose_dialog(dname='', level=1, include='all', strip_private=False, fmt='text'):
    "Disclose dialog `dname` at `level` filtering by `include`"
    srcs = await dialog_code(dname=dname, include=include)
    return _disclose_srcs(srcs, level=level, strip_private=strip_private, fmt=fmt, dname=dname)

def disclose_nb(path, level=1, strip_private=False, fmt='text'):
    "Return disclosed source of all code cells in notebook at `path`"
    nb = json.loads(Path(path).read_text())
    srcs = [(c.get('id',''), c['source'] if isinstance(c['source'], str) else ''.join(c['source']))
            for c in nb['cells'] if c['cell_type'] == 'code']
    return _disclose_srcs(srcs, level=level, strip_private=strip_private, fmt=fmt, dname=str(path))

### tests

In [ ]:
def _test_dlg(cs, fp='test_disclose.ipynb'):
    nb = writes_json(new_notebook(cells=[new_code_cell(c) for c in cs]))
    (p := Path(fp)).write_text(nb)
    return str(p)

test_eq(disclose('', 1), '')
test_eq(disclose('', 2), '')
test_eq(disclose('', 3), '')

test_eq(disclose('(1+2)', 1), '')
test_eq(disclose('(1+2)', 2), '')
test_eq(disclose('(1+2)', 3), '')

src = 'if a: import A\nelse: import B'
test_eq(disclose(src, 1), src)
test_eq(disclose(src, 2), src)
test_eq(disclose(src, 3), src)

_s_mods = "x = 5\ny:int = 10\ntype Alias = list[int]"
test_eq(disclose(_s_mods, 1), 'x = 5\n\ny:int = 10\n\ntype Alias = list[int]')
test_eq(disclose(_s_mods, 2), disclose(_s_mods, 1))
test_eq(disclose(_s_mods, 3), disclose(_s_mods, 1))

In [ ]:
_s_moddoc = '"""This module does X."""\n\ndef foo(): pass'
test_eq(disclose(_s_moddoc, 1), "def foo(): ...")
test_eq(disclose(_s_moddoc, 2), '"""This module does X."""\n\ndef foo(): ...')
test_eq(disclose(_s_moddoc, 3), '"""This module does X."""\n\ndef foo(): pass')

_s_fn = "def foo(a:int, b:str='hi') -> bool: return True"
test_eq(disclose(_s_fn, 1), "def foo(a: int, b: str='hi') -> bool: ...")
test_eq(disclose(_s_fn, 3), _s_fn)

_s_async = "async def fetch(url:str) -> bytes:\n    'Fetch url.'\n    pass"
test_eq(disclose(_s_async, 1), "async def fetch(url: str) -> bytes: ...")

_s_deco1 = "@patch\ndef foo(self:MyClass, x:int): pass"
test_eq(disclose(_s_deco1, 1), "@patch\ndef foo(self: MyClass, x: int): ...")

_s_deco2 = "@log\n@patch\ndef foo(self:MyClass, x:int): pass"
test_eq(disclose(_s_deco2, 1), "@log\n@patch\ndef foo(self: MyClass, x: int): ...")

_s_doc = '\ndef add(\n    a:int, # first\n    b:int=0, # second\n) -> int:\n    "Sum two numbers."\n    return a+b\n'
test_eq(disclose(_s_doc, 1), "def add(a: int, b: int=0) -> int: ...")
test_eq(disclose(_s_doc, 2), 'def add(\n    a:int, # first\n    b:int=0, # second\n) -> int:\n    "Sum two numbers."\n    ...')
test_eq(disclose(_s_doc, 3), _s_doc.strip())
test_is('return a+b' in disclose(_s_doc, 2), False)
test_is('return a+b' in disclose(_s_doc, 3), True)

In [ ]:
_s_cls = '\nclass Foo(Base):\n    "A foo."\n    def __init__(self,\n                 x:int, # x val\n                 y:str=\'hi\'): # y val\n        "Init."\n        self.x,self.y = x,y\n    def bar(self) -> bool:\n        "Check bar."\n        return self.x > 0\n    def _priv(self): pass\n'
_cls_l1 = 'class Foo(Base):\n    def __init__(self, x: int, y: str=\'hi\'): ...\n    def bar(self) -> bool: ...\n    def _priv(self): ...'
_cls_l2 = 'class Foo(Base):\n    "A foo."\n    def __init__(self,\n                 x:int, # x val\n                 y:str=\'hi\'): # y val\n        "Init."\n        ...\n    def bar(self) -> bool:\n        "Check bar."\n        ...\n    def _priv(self): ...'
test_eq(disclose(_s_cls, 1), _cls_l1)
test_eq(disclose(_s_cls, 2), _cls_l2)
test_eq(disclose(_s_cls, 3), _s_cls.strip())

_s_cls_assigns = '\nclass Config:\n    "App config."\n    debug: bool = False\n    MAX_RETRIES: int = 3\n    name = "default"\n    def reset(self): self.debug = False\n'
test_eq(disclose(_s_cls_assigns, 1), 'class Config:\n    debug: bool = False\n    MAX_RETRIES: int = 3\n    name = "default"\n    def reset(self): ...')
test_eq(disclose(_s_cls_assigns, 2), 'class Config:\n    "App config."\n    debug: bool = False\n    MAX_RETRIES: int = 3\n    name = "default"\n    def reset(self): ...')
test_eq(disclose(_s_cls_assigns, 3), _s_cls_assigns.strip())

_s_dc = '\nfrom dataclasses import dataclass\n\n@dataclass\nclass Point:\n    "A 2D point."\n    x: float\n    y: float = 0.0\n    def dist(self) -> float:\n        "Distance from origin."\n        return (self.x**2 + self.y**2)**0.5\n'
test_eq(disclose(_s_dc, 1), "@dataclass\nclass Point:\n    x: float\n    y: float = 0.0\n    def dist(self) -> float: ...")
test_eq(disclose(_s_dc, 2), "from dataclasses import dataclass\n\n@dataclass\nclass Point:\n    \"A 2D point.\"\n    x: float\n    y: float = 0.0\n    def dist(self) -> float:\n        \"Distance from origin.\"\n        ...")
test_eq(disclose(_s_dc, 3), _s_dc.strip())

_s_cond = '\nimport sys\n\nif sys.version_info >= (3, 12):\n    def new_feature(x:int) -> str: return str(x)\nelse:\n    def new_feature(x:int) -> str: return repr(x)\n\ndef always_here(): pass\n'
test_eq(disclose(_s_cond, 1), 'if sys.version_info >= (3, 12):\n    def new_feature(x: int) -> str: ...\nelse:\n    def new_feature(x: int) -> str: ...\n\ndef always_here(): ...')
test_eq(disclose(_s_cond, 2), 'import sys\n\nif sys.version_info >= (3, 12):\n    def new_feature(x: int) -> str: ...\nelse:\n    def new_feature(x: int) -> str: ...\n\ndef always_here(): ...')
test_eq(disclose(_s_cond, 3), _s_cond.strip())

In [ ]:
_s_imp = '\n"""A module."""\nimport os, sys\nfrom pathlib import Path\n\ndef foo(): pass\n'
test_eq(disclose(_s_imp, 1), "def foo(): ...")
test_eq(disclose(_s_imp, 2), '"""A module."""\n\nimport os, sys\n\nfrom pathlib import Path\n\ndef foo(): ...')
test_eq(disclose(_s_imp, 3), '"""A module."""\n\nimport os, sys\n\nfrom pathlib import Path\n\ndef foo(): pass')

_s_tricky = '''
def foo():
    """
    Example::

        :\\n    ...

    ```python
    x = 1
    ```
    """
    pass

def bar(): pass

def baz():
    """Has "quotes" and \\'escaped\\' stuff."""
    pass
'''
test_eq('```python' in disclose(_s_tricky, 2), True)
test_eq('pass' in disclose(_s_tricky, 3), True)

_test_sp = '\ndef pub(x:int) -> int:\n    "public fn"\n    return _helper(x)\n\ndef _helper(x): return x+1\n\nclass Foo:\n    def pub_method(self): return 1\n    def _priv_method(self): return 2\n'
test_eq(disclose(_test_sp, 1, strip_private=True), 'def pub(x: int) -> int: ...\n\nclass Foo:\n    def pub_method(self): ...')
test_eq(disclose(_test_sp, 2, strip_private=True), 'def pub(x:int) -> int:\n    "public fn"\n    ...\n\nclass Foo:\n    def pub_method(self): ...')

cs = ['a=10', 'def f(a:int # arg\n) -> int: # ret\n    "a doc"\n    return a+20']
test_dname = _test_dlg(cs)
test_eq(disclose_nb(test_dname, level=1), 'a=10\ndef f(a: int) -> int: ...')
test_eq(disclose_nb(test_dname, level=2), 'a=10\ndef f(a:int # arg\n) -> int: # ret\n    "a doc"\n    ...')
test_eq(disclose_nb(test_dname, level=3), 'a=10\ndef f(a:int # arg\n) -> int: # ret\n    "a doc"\n    return a+20')

#| export
### Code disclosure: rationale, decisions, current state

Progressive disclosure of Python source code at three levels modelled on `.pyi` stub files. Per-cell, preserving message boundaries and dialog narrative structure.

| Level | Description | Analogy |
|---|---|---|
| 1 | Sigs only, no docs, no imports, no comments | Minified `.pyi` |
| 2 | Sigs + docstrings + inline param comments + imports | Standard `.pyi` |
| 3 | Full source, untouched | `.py` |

**Implementation**: `asttokens` maps AST nodes to original source spans. Level 1 uses `ast.unparse` (comments stripped naturally). Level 2 lifts original source lines (comments preserved). `_postprocess` hoists lone `...` bodies and collapses triple+ blank lines.

**Design decisions**:
- Imports stripped at level 1 only (executability at levels 2/3)
- Bare `ast.Expr` stripped at all levels except module docstring at level ≥ 2
- `strip_private` flag removes `_`-prefixed functions/methods (safe at levels 1/2)
- `_compact` collapses inter-cell blank lines for text output format
- Three inclusion tiers via `_include_filter`: `'exported'`, `'echo'`, `'all'`

**Known limitations**:
- `_postprocess` ellipsis hoisting can match inside docstrings containing `:\n    ...` patterns (rare)
- Side-effect expressions (`print(...)`, `setup(...)`) are stripped — indistinguishable from pure expressions by AST alone
- `_compound_chunk` uses `text.replace()` which could misfire if two functions inside a compound have identical source text
- `@patch` consolidation and `@delegates` expansion not yet implemented

**Token ratios** (this dialog, exported cells): level 1 ~22%, level 2 ~43%, level 3 ~105% of raw.

## branching
1. ephemeral -> git worktree?
2. permanent
How to do dialog branching:
- simple, no tracking: dup dialog -> create new dialog with no references in resulting dialogs.
- simple, with origin tracking: dup dialog -> the new dialog point to the origin. Fragile: origin can change and stale the chain.
- involved, git tracking: clean status (persistent commit before branching): branch stores the commit hash, we can reconstruct history or even attempt merging/syncing.

Great! Ok, what next? Be wild!

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

A few directions, roughly by ambition:

1. **Content search** — `contains='pattern'` regex filter on message content. Simple, composable, immediately useful for "find all cells referencing symbol `foo`".

2. **Token budget** — `max_tokens=N`: take messages in dialog order until budget exhausted. Critical for fitting digests into LLM context windows without manual slicing.

3. **Wire into `exportmsgs`** — accept `filter_msgs_v1` kwargs directly so callers don't need to pre-filter; deprecate `filter_msgs_v0`.

4. **Dependency tracking** — given a set of code cells, walk backwards through the dialog to find all cells that define symbols they reference. True isolation: a slice that can run correctly in a fresh kernel. This is the wildest one — essentially a lightweight static analyser over dialog cells.

The last one is the most powerful and would unlock real "extract section to new dialog" workflows. But #1 or #2 are good warm-ups.

<details class='token-usage-details'><summary>$0.0512</summary>

`total=44,318 | in=43,762 | out=556 | cached=91.5% | cache_new=3,458 | searches=0 | $0.0512`

</details>


## dependency tracking\isolation
It is an important goal. It is in my list and I got first steps with my isolated dialog (prj/pote/explorer/isolated/isolated) and my dutil.flakes module. We'll tackle that later (probably in another session).

## import

Brings messages from other dialogs into the current one — selecting by symbol name, message id, or regex — and optionally runs them on insertion.

see <a class="uk-link" href="/dialog_?name=prj%2Fdutil%2Fexplorer%2Fimportdlg%2Fimportdlg" target="_blank">importdlg exploratory dialog</a>

In [ ]:
#| export
def _r(m:dict, **kwargs):
    "Add kwargs to m and filter out key/value not compatible with add_msg or update_msg"
    if kwargs: m.update(kwargs)
    if m.msg_type == 'code': m.output = ''
    return dict(content=m.content, msg_type=m.msg_type, output=m.output, is_exported=m.is_exported, skipped=m.skipped, pinned=m.pinned, i_collapsed=m.i_collapsed, o_collapsed=m.o_collapsed, heading_collapsed=m.heading_collapsed)

In [ ]:
#| export
def find_symbol_msg(msgs, sym:str) -> dict|None:
    "Find last message in msgs list containing symbol definition (module level)"
    for m in reversed(msgs):
        if m.msg_type != 'code': continue
        root = ast_py(m.content)
        for node in root.find_all(pattern='def $FUNC($$$)'):
            if node.field('name').text() == sym: return m
        for node in root.find_all(pattern='class $CLASS'):
            if node.field('name').text() == sym: return m
        for node in root.find_all(pattern='$VAR = $$$'):
            if node.field('left').text() == sym: return m
    return None

In [ ]:
#| export
async def importdlg(
    dname:str='',  # dialog to import from; relative to solveit root (if starts with /) or current dialog; empty string for current dialog
    syms:list[str]=None,  # symbol names to import (finds last def/class/assignment); if provided, regex is ignored
    ids:list[str]=None,  # message ids to import; can combine with syms; if syms or ids provided, regex is ignored
    re_include:str=None,  # regex to filter messages (content+output); only used if no syms/ids; uses re.MULTILINE|re.DOTALL
    re_exclude:str=None,  # regex to exclude from re_include results; only used if no syms/ids
    run:bool=False,  # run imported messages after insertion
    header:bool=False,  # wrap imports in collapsed header with dialog name
    heading_collapsed:bool=False  # collapse the header after creation
):
    "Import messages from dname by symbols, ids, regex filter, or all messages; inserts below current message in original order"
    msgs, to_import = await find_msgs(dname=dname, include_meta=True, include_output=True), []
    if not dname: msgs = msgs[:await msg_idx()]
    if syms or ids:
        id2idx, seen = {m.id:i for i,m in enumerate(msgs)}, set()
        if syms:
            for sym in syms:
                if (m := find_symbol_msg(msgs, sym)) and m.id not in seen: to_import.append(m); seen.add(m.id)
        if ids:
            for m in msgs:
                if m.id in ids and m.id not in seen: to_import.append(m); seen.add(m.id)
        to_import.sort(key=lambda m: id2idx[m.id])
    else:
        to_import = msgs
        if re_include: to_import = [m for m in to_import if re.search(re_include, m.content + m.output, re.MULTILINE | re.DOTALL)]
        if re_exclude: to_import = [m for m in to_import if not re.search(re_exclude, m.content + m.output, re.MULTILINE | re.DOTALL)]
    new_ids = [await add_msg(**_r(m)) for m in to_import]
    if header and new_ids: 
        hdrid = await add_msg(f"## Imported from {dname}", placement='add_before', id=new_ids[0])
        new_ids.insert(0, hdrid)
        new_ids.append(await add_msg("## ----", id=new_ids[-1]))
    # time.sleep(0.5)
    if run and new_ids: await run_msg(','.join(new_ids))
    # time.sleep(0.1)
    if heading_collapsed and header: await toggle_header(hdrid)
    return new_ids

## export -

In [ ]:
from pote.dutil import ctxusage
await ctxusage()

97207

In [ ]:
from pote.flakes import show_flakes
await show_flakes()

<div class="prose">

No warnings to report

</div>

In [ ]:
# #|hide
# #|eval: false
# from pote.dialog import dlg_export
# await dlg_export()